# Expert Spark Interview Preparation — Principal Architect Level

**Audience:** Spark Architects, Principal Engineers, Staff Engineers

**Dataset:** NYC Taxi Cab Registry (~8,970 rows) — `Cabs.csv`

**Topics Covered:**
1. Catalyst Optimizer (rule-based + cost-based, plan transformations)
2. Tungsten Engine (WSCG, UnsafeRow, vectorization)
3. Cost-Based Optimizer (CBO, statistics, join reordering)
4. Join Strategy Selection (all 5 join types, hints, AQE interaction)
5. Cluster Sizing (formulas, parallelism, resource math)
6. Executor Tuning (fat vs thin, GC, off-heap)
7. End-to-End Pipeline Optimization (10-step audit framework)
8. Spark UI Deep Dive (every tab, every metric)

---
**Reference:** Spark source packages: `org.apache.spark.sql.catalyst`, `org.apache.spark.sql.execution`, `org.apache.spark.unsafe`

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import time, tempfile, json

active = SparkSession.getActiveSession()
if active: active.stop()

spark = SparkSession.builder \
    .appName("SparkInterviewPrep-Expert") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.cbo.enabled", "true") \
    .config("spark.sql.statistics.histogram.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
CABS_PATH = r"C:\Users\PS\Documents\Python-Exp\RawData\Cabs.csv"
TEMP_DIR = tempfile.mkdtemp()
print("Spark version:", spark.version)
print("Temp dir:", TEMP_DIR)

## Topic 1 — Catalyst Optimizer

### Scenario
You are a Principal Engineer at a ride-sharing company. Your team's Spark job reads the full NYC Cab Registry (~8,970 rows, 14 columns), applies multiple filters, selects a subset of columns, and joins to a license type lookup table. The job runs in 45 seconds. A junior engineer asks why the query plan looks different from the code they wrote. You must explain the full Catalyst pipeline.

### Dataset
- **Source:** Cabs.csv (~8,970 rows)
- **Schema:** CabNumber, VehicleLicenseNumber, Name, LicenseType, Active, PermitLicenseNumber, VehicleVinNumber, WheelchairAccessible, VehicleYear, VehicleType, TelephoneNumber, Website, Address, LastDateUpdated
- **Volume simulation:** Broadcast 10x to simulate 90K rows

### Catalyst 4-Phase Architecture
```
SQL / DataFrame API
        |
        v
   [Phase 1] Analysis
   - Resolve column references via Catalog
   - Type coercion (ImplicitTypeCasts)
   - Produces: Analyzed Logical Plan
        |
        v
   [Phase 2] Logical Optimization  <-- Rule[LogicalPlan] batch application
   - ConstantFolding
   - PredicatePushdown
   - ColumnPruning
   - CombineFilters
   - Produces: Optimized Logical Plan
        |
        v
   [Phase 3] Physical Planning  <-- SparkPlanner applies Strategy objects
   - Multiple physical plans generated
   - Cost model selects best (with CBO)
   - Produces: SparkPlan (Physical Plan)
        |
        v
   [Phase 4] Code Generation  <-- WholeStageCodegenExec wraps plan
   - Janino compiles Java bytecode at runtime
   - Tight loop: no virtual dispatch, no boxing
   - Produces: Executable RDD
```

### Key Spark Source References
- `org.apache.spark.sql.catalyst.rules.Rule[LogicalPlan]` — base class for all optimizer rules
- `org.apache.spark.sql.catalyst.trees.TreeNode` — AST node abstraction; every LogicalPlan/Expression is a TreeNode
- `org.apache.spark.sql.catalyst.optimizer.Optimizer` — applies rule batches via `executeAndTrack`
- `org.apache.spark.sql.execution.SparkPlanner` — converts LogicalPlan → SparkPlan
- `org.apache.spark.sql.execution.WholeStageCodegenExec` — wraps physical operators that support `CodegenSupport`

In [ ]:
# ── Topic 1: Data Setup ──────────────────────────────────────────────────────
cabs_schema = StructType([
    StructField("CabNumber",              StringType(),  True),
    StructField("VehicleLicenseNumber",   StringType(),  True),
    StructField("Name",                   StringType(),  True),
    StructField("LicenseType",            StringType(),  True),
    StructField("Active",                 StringType(),  True),
    StructField("PermitLicenseNumber",    StringType(),  True),
    StructField("VehicleVinNumber",       StringType(),  True),
    StructField("WheelchairAccessible",   StringType(),  True),
    StructField("VehicleYear",            IntegerType(), True),
    StructField("VehicleType",            StringType(),  True),
    StructField("TelephoneNumber",        StringType(),  True),
    StructField("Website",                StringType(),  True),
    StructField("Address",                StringType(),  True),
    StructField("LastDateUpdated",        StringType(),  True),
])

cabs_raw = spark.read.csv(CABS_PATH, header=True, schema=cabs_schema)
# Simulate larger dataset
cabs = cabs_raw
for _ in range(3):          # ~72K rows
    cabs = cabs.union(cabs_raw)

cabs.createOrReplaceTempView("cabs")
print(f"Row count: {cabs.count():,}")

# Small lookup table — perfect broadcast candidate
license_lookup = spark.createDataFrame([
    ("OWNER MUST DRIVE",  "O",  "Owner-operated vehicle"),
    ("NAMED DRIVER",      "N",  "Employee driver"),
    ("OWNER NAMED DRIVER","ON", "Owner who also drives"),
], ["LicenseType", "LicenseCode", "Description"])
license_lookup.createOrReplaceTempView("license_lookup")
print("License lookup rows:", license_lookup.count())

### Problem Statement
The code below has multiple redundant operations that Catalyst should optimize. After running the 'poorly optimized' code, inspect each phase of the query execution plan to see what Catalyst actually does versus what was written.

In [ ]:
# ── Poorly Optimized: Anti-patterns Catalyst must fix ────────────────────────
# Anti-pattern 1: Redundant filters on same column (should be CombineFilters)
# Anti-pattern 2: Selecting all columns, then selecting fewer (ColumnPruning needed)
# Anti-pattern 3: Constant expression in filter (ConstantFolding)
# Anti-pattern 4: Filter AFTER join instead of BEFORE (PredicatePushdown)

df_bad = (
    cabs
    .select("*")                          # selects all 14 columns unnecessarily
    .filter(F.col("VehicleYear") > 2000)  # filter 1 — will be pushed down
    .filter(F.col("VehicleYear") > 2010)  # filter 2 — redundant, subsumes filter 1
    .filter(F.lit(1) == F.lit(1))         # filter 3 — constant TRUE, should be eliminated
    .join(license_lookup, "LicenseType", "left")
    .filter(F.col("Active") == "YES")     # should be pushed BEFORE the join
    .select("CabNumber", "Name", "LicenseCode", "VehicleYear", "Active")
)

print("=" * 80)
print("ANALYZED LOGICAL PLAN (after name resolution, before optimization):")
print("=" * 80)
print(df_bad.queryExecution.analyzed)

print("\n" + "=" * 80)
print("OPTIMIZED LOGICAL PLAN (after Catalyst rule batches):")
print("=" * 80)
print(df_bad.queryExecution.optimizedPlan)

print("\n" + "=" * 80)
print("PHYSICAL PLAN (SparkPlanner output, what actually executes):")
print("=" * 80)
print(df_bad.queryExecution.executedPlan)

### Interview Questions — Topic 1: Catalyst Optimizer

1. **TreeNode & Pattern Matching:** Explain how Catalyst uses Scala's pattern matching on `TreeNode` subtypes to apply optimization rules. What is the difference between `transformDown` and `transformUp` on a tree? When does the order of traversal matter?

2. **Rule Batches & Fixed Points:** Catalyst applies rules in *batches* with a *fixed-point* iteration (runs until plan stops changing) or *once* mode. Name three rules that run once-only and explain why running them to fixed point would be incorrect or dangerous.

3. **PredicatePushdown Limits:** PredicatePushdown does not always push filters through all operators. Name three LogicalPlan node types through which a filter CANNOT be pushed, and explain why each one blocks pushdown.

4. **Analysis Phase Failures:** A query runs fine in isolation but throws `AnalysisException` inside a Spark job. What are the five most common causes of AnalysisException and how would you debug each in production without a REPL?

5. **Custom Optimizer Rules:** Walk me through writing a custom `Rule[LogicalPlan]` that rewrites `UPPER(LOWER(col))` to just `UPPER(col)`. What class do you extend? Where do you inject the rule? What are the risks of injecting rules into early vs late batches?

6. **Subquery Handling:** How does Catalyst handle correlated subqueries (e.g., `WHERE EXISTS (SELECT ...)`)? What plan transformations convert them to joins? What is `DecorrelateInnerQuery` and when does it fail?

7. **AQE vs Catalyst:** Adaptive Query Execution (AQE) re-optimizes the plan at runtime after each shuffle. How does AQE interact with Catalyst's static optimizer? What plan information does AQE use that Catalyst cannot access at compile time?

8. **CBO vs Rule-Based:** In a query joining four tables, Catalyst's rule-based optimizer always puts the largest table first in a join. CBO reorders joins. Describe the exact mechanism CBO uses — what statistics does it collect, what dynamic programming algorithm does it apply, and what is `spark.sql.cbo.joinReorder.dp.threshold`?

### Expected Logical Plan Discussion

**Analyzed Logical Plan** (before optimization) will show:
```
Project [CabNumber, Name, LicenseCode, VehicleYear, Active]
+- Filter (Active = YES)
   +- Join LeftOuter, (LicenseType = LicenseType)
      :- Filter ((1 = 1) AND (VehicleYear > 2010) AND (VehicleYear > 2000))
      :  +- Project [*]  <-- all 14 columns
      :     +- Relation cabs
      +- Relation license_lookup
```

**Optimized Logical Plan** — Catalyst applies these rules in order:

| Rule | Effect | Batch |
|------|--------|-------|
| `ConstantFolding` | Eliminates `1 = 1` → always true, filter removed | Operator Optimization |
| `CombineFilters` | Merges `VehicleYear > 2000` AND `VehicleYear > 2010` into single Filter | Operator Optimization |
| `InferFiltersFromConstraints` | `VehicleYear > 2010` subsumes `> 2000`, removes redundant | Operator Optimization |
| `PushDownPredicates` | Moves `Active = YES` filter below the Join to the left side | Push Down Predicates |
| `ColumnPruning` | Removes `Project [*]`, only reads columns needed downstream | Operator Optimization |

**Result:** The optimized plan reads only 5 columns, applies both filters before the join, and eliminates the constant filter — all without the developer changing a single line of code.

**Why this matters in production:** On a 10TB dataset with 200 columns, ColumnPruning alone reduces I/O by 90%+ when reading Parquet (which is columnar). PredicatePushdown into Parquet's row-group statistics means entire file blocks are skipped at the storage layer.

### Expected Physical Plan Discussion

**SparkPlanner Strategies** (applied in order, first match wins):
1. `SpecialLimits` — handles LIMIT
2. `Aggregation` — Hash vs Sort aggregation
3. `JoinSelection` — selects join algorithm (BroadcastHashJoin for small tables)
4. `InMemoryScans` — cached relation
5. `BasicOperators` — Filter, Project, Sort, etc.

**For this query, Physical Plan shows:**
```
*(2) Project [CabNumber, Name, LicenseCode, VehicleYear, Active]
+- *(2) BroadcastHashJoin [LicenseType], [LicenseType], LeftOuter, ...
   :- *(2) Filter (Active = YES AND VehicleYear > 2010)
   :  +- *(2) FileScan csv [CabNumber,Name,LicenseType,Active,VehicleYear]
   +- BroadcastExchange HashedRelationBroadcastMode
      +- *(1) LocalTableScan [LicenseType, LicenseCode, Description]
```

**Key observations:**
- `*(2)` prefix means WholeStageCodegen stage 2 — these operators compile to a single Java method
- `BroadcastHashJoin` chosen because `license_lookup` is 3 rows (well under `autoBroadcastJoinThreshold` = 10MB)
- `FileScan` shows only 5 columns — ColumnPruning worked at the scan level
- Filter is applied AT the scan — PredicatePushdown worked
- `BroadcastExchange` ships the small table to all executors — zero shuffle on the large side

### Spark UI Investigation Guide — Catalyst Plan Inspection

**SQL/DataFrame Tab** (`http://<driver>:4040/SQL`):
- Click on the query → see the DAG visualization
- Each box = one Exchange (shuffle) or scan boundary
- Look for **"WholeStageCodegen"** labels — good (tight loop)
- Look for **"Exchange"** nodes — each = a shuffle = network I/O

**Plan Details Panel** (click "Details" on SQL tab):
```
== Parsed Logical Plan ==      ← raw AST from parser
== Analyzed Logical Plan ==    ← after catalog resolution
== Optimized Logical Plan ==   ← after Catalyst rule application
== Physical Plan ==            ← what actually runs
```

**Warning signs in the plan:**
- `CartesianProduct` — missing join condition, will OOM on large data
- `SortMergeJoin` where you expected `BroadcastHashJoin` — table exceeded threshold or stats missing
- `Exchange` between two `Filter` nodes — filter not pushed down (check for non-deterministic UDF blocking pushdown)
- No `*(N)` prefix on operators — WholeStageCodegen disabled, check for UDFs or unsupported operators
- `ObjectHashAggregate` instead of `HashAggregate` — using Python UDFs in aggregation

In [ ]:
# ── Topic 1: Optimization Exercise ──────────────────────────────────────────
# Show the CORRECT way — let Catalyst work, but also guide it properly
# Use explain(True) to see all 4 plan stages

df_optimized = (
    cabs
    .select("CabNumber", "Name", "LicenseType", "Active", "VehicleYear")  # explicit columns
    .filter((F.col("VehicleYear") > 2010) & (F.col("Active") == "YES"))   # single combined filter
    .join(F.broadcast(license_lookup), "LicenseType", "left")              # explicit broadcast hint
    .select("CabNumber", "Name", "LicenseCode", "VehicleYear", "Active")
)

print("OPTIMIZED — explain(extended=True) shows all 4 plan phases:")
df_optimized.explain(extended=True)

In [ ]:
# ── Topic 1: Performance Benchmarking ────────────────────────────────────────
import time

def timed_count(df, label):
    start = time.time()
    n = df.count()
    elapsed = time.time() - start
    print(f"{label}: {n:,} rows in {elapsed:.2f}s")
    return elapsed

# Warm up
_ = cabs.count()

t_bad  = timed_count(df_bad,       "Bad  (Catalyst still optimizes, but we fight it)")
t_good = timed_count(df_optimized, "Good (aligned with Catalyst)")
print(f"\nSpeedup: {t_bad/t_good:.2f}x  (Catalyst closes most of the gap automatically)")

# Show the rules Catalyst applied (via Java reflection on the optimizer)
print("\nCatalyst optimizer rule batches applied:")
for batch in spark._jsparkSession.sessionState().optimizer().batches():
    rules = [r.ruleName() for r in batch.rules()]
    print(f"  Batch '{batch.name()}': {len(rules)} rules")
    if len(rules) < 8:  # only print short batches in full
        for r in rules:
            print(f"    - {r}")

### Production Best Practices — Catalyst Optimizer

1. **Always inspect the Physical Plan before promoting code to production.** Use `df.explain(extended=True)` or the Spark UI SQL tab. The optimized plan is what actually runs — not your Python code.

2. **Never use `select("*")` in production pipelines.** Even though ColumnPruning will trim unused columns, explicit column selection documents intent, prevents schema drift bugs, and avoids analysis overhead when schemas change.

3. **Push filters as early as possible in your code, even though Catalyst will do it.** Code that is already filtered early is more readable, and some data sources (JDBC, custom connectors) require explicit predicates to push filters to the source system — Catalyst cannot always do this automatically.

4. **Non-deterministic functions block PredicatePushdown.** `F.rand()`, `F.randn()`, `F.monotonically_increasing_id()` — if used in the WHERE clause or in a column referenced by a filter, pushdown stops. Audit all uses of non-deterministic UDFs in filter paths.

5. **Custom optimizer rules belong in a Spark Extension, not monkey-patched into the session.** Use `SparkSessionExtensions.injectOptimizerRule()` for safe injection. Document the batch position carefully — rules injected after `Join Reorder` will not affect join ordering.

6. **Catalyst cannot optimize through Python UDFs.** The optimizer treats UDFs as black boxes. A filter on a UDF result CANNOT be pushed below the UDF. This is a hard boundary — no workaround except rewriting the UDF as a native Spark expression.

7. **ANALYZE TABLE before any CBO-dependent optimization.** CBO join reordering requires accurate row counts and column NDV (number of distinct values). Without `ANALYZE`, Catalyst falls back to rule-based ordering which may produce suboptimal plans on multi-way joins.

8. **Monitor plan regressions on Spark version upgrades.** Catalyst rule batches change between Spark versions. A plan that used BroadcastHashJoin in 3.2 might switch to SortMergeJoin in 3.4 if a new rule changes threshold evaluation. Always re-explain plans after upgrading.

9. **AQE's `coalescePartitions` and `convertJoinToLocalSort` fire AFTER Catalyst.** AQE overrides static optimizer decisions at runtime. If you see unexpected join strategies in the UI that differ from `explain()`, AQE changed the plan at execution time. Check `spark.sql.adaptive.enabled` and `spark.sql.adaptive.localShuffleReader.enabled`.

10. **`queryExecution.stringWithStats` shows plan with row estimates.** This is the CBO-annotated plan. If you see `sizeInBytes=8.0 EiB` (default unknown size), CBO has no statistics and is guessing. Run ANALYZE or provide explicit broadcast hints.

### Follow-up Questions — Topic 1

- How does Catalyst handle view expansion? If a view references another view, how many levels of plan inlining does Catalyst perform, and is there a limit?
- What is `EliminateSubqueryAliases` and when is it applied? What happens if you have 50 nested subqueries in SQL?
- Explain the difference between `LogicalPlan.resolved` and `LogicalPlan.analyzed`. When would a plan be analyzed but not resolved?
- In Databricks Runtime (DeltaEngine), Delta Lake adds custom optimizer rules (e.g., `OptimizeMetadataOnlyQuery`, `PreprocessTableUpdate`). How are vendor-specific rules injected without modifying Spark core?

In [ ]:
# ── Topic 1: Expert Solution — Catalyst Plan Introspection Utility ───────────
# Production-grade utility to audit any DataFrame's optimization

def audit_catalyst_plan(df, label="DataFrame"):
    """
    Full Catalyst plan audit.
    In production: log this to your observability platform before executing.
    Detect anti-patterns: CartesianProduct, missing pushdown, WSCG disabled.
    """
    qe = df.queryExecution
    physical_str = str(qe.executedPlan)

    warnings = []

    # Anti-pattern detection via string matching on physical plan
    if "CartesianProduct" in physical_str:
        warnings.append("CRITICAL: CartesianProduct detected — missing join condition")
    if "SortMergeJoin" in physical_str and "BroadcastExchange" not in physical_str:
        # Not necessarily bad, but flag for review when small tables are joined
        warnings.append("INFO: SortMergeJoin in use — verify both sides need shuffle")
    if "ObjectHashAggregate" in physical_str:
        warnings.append("WARNING: ObjectHashAggregate — Python UDAF or complex type bypassing Tungsten")
    if "Generate" in physical_str and "explode" in physical_str.lower():
        warnings.append("INFO: explode() detected — verify input cardinality won't cause data explosion")

    # Count WholeStageCodegen stages
    wscg_stages = physical_str.count("WholeStageCodegen")
    if wscg_stages == 0:
        warnings.append("WARNING: No WholeStageCodegen stages — all operators are interpreted (slow)")

    # Count Exchange (shuffle) operations
    exchanges = physical_str.count("Exchange")

    # Report
    print(f"\n{'='*70}")
    print(f"Catalyst Plan Audit: {label}")
    print(f"{'='*70}")
    print(f"  WholeStageCodegen stages : {wscg_stages}")
    print(f"  Exchange (shuffle) ops   : {exchanges}")
    print(f"  Optimized plan nodes     : {len(qe.optimizedPlan.collect())}")
    print(f"  Analyzed plan nodes      : {len(qe.analyzed.collect())}")
    plan_reduction = (1 - len(qe.optimizedPlan.collect()) / max(1, len(qe.analyzed.collect()))) * 100
    print(f"  Plan node reduction      : {plan_reduction:.1f}% (Catalyst eliminated nodes)")
    if warnings:
        print(f"\n  Warnings ({len(warnings)}):")
        for w in warnings:
            print(f"    [{w.split(':')[0]}] {':'.join(w.split(':')[1:]).strip()}")
    else:
        print("\n  No anti-patterns detected.")
    print()

audit_catalyst_plan(df_bad,       "Bad Query")
audit_catalyst_plan(df_optimized, "Optimized Query")

## Topic 2 — Tungsten Engine

### Scenario
Your team's ML feature engineering pipeline runs Spark on a 50-node cluster but CPU utilization is only 35%. A profiling session reveals that 40% of task time is spent in JVM garbage collection. A Python UDF for string normalization is in the hot path. You must redesign the pipeline around Tungsten's execution model.

### Dataset
- Source: Cabs.csv (~8,970 rows × 14 cols)
- Simulation: 5× union to produce ~44,850 rows — small enough to iterate fast, large enough to see GC/codegen differences
- Focus: string processing (Name, Address) and numeric processing (VehicleYear)

### Tungsten Architecture Deep Dive
```
Project Tungsten (Spark 1.4+) — three pillars:

1. MEMORY MANAGEMENT
   ┌──────────────────────────────────────────────────────┐
   │  JVM Heap (on-heap)      Off-Heap (unsafe memory)   │
   │  ┌────────────────┐     ┌──────────────────────────┐│
   │  │ Java objects   │     │ UnsafeRow binary format  ││
   │  │ GC overhead    │     │ No GC — manual alloc/free││
   │  │ Object headers │     │ sun.misc.Unsafe.allocate ││
   │  │ 16B per obj    │     │ Contiguous memory pages  ││
   │  └────────────────┘     └──────────────────────────┘│
   └──────────────────────────────────────────────────────┘

2. CACHE-AWARE COMPUTATION
   UnsafeRow layout: [null_bitmap | fixed_len_fields | var_len_data]
   All fixed-length fields are 8 bytes (long). Strings → offset+length in
   fixed section, data in var-len section. Row fits in CPU cache lines.

3. WHOLE-STAGE CODE GENERATION (WSCG)
   Instead of: iterator.hasNext() + virtualDispatch() per operator per row
   Generates:  single tight Java loop via Janino compiler
   
   // Generated Java (conceptual):
   while (inputIter.hasNext()) {
     InternalRow row = inputIter.next();
     if (row.getInt(8) > 2010 && row.getString(4).equals("YES")) {
       // all operators fused — no virtual dispatch
       output(row.getString(0), row.getString(2), row.getInt(8));
     }
   }
```

### sun.misc.Unsafe — The JVM Escape Hatch
- `Unsafe.allocateMemory(bytes)` — allocates native memory outside GC heap
- `Unsafe.putLong(address, value)` — writes directly to memory address
- `Unsafe.getLong(address)` — reads directly from memory address
- `Unsafe.copyMemory(src, dst, bytes)` — SIMD-friendly memory copy
- Why Spark uses it: serialize/deserialize an UnsafeRow = pointer arithmetic, no object creation, no GC pressure
- Risk: segfault on bad pointer. Spark wraps all unsafe ops in bounds-checked `Platform` class.

In [ ]:
# ── Topic 2: Data Setup ──────────────────────────────────────────────────────
cabs_t2 = cabs_raw
for _ in range(4):    # ~44,850 rows
    cabs_t2 = cabs_t2.union(cabs_raw)
cabs_t2.cache()
print(f"Topic 2 dataset: {cabs_t2.count():,} rows")

# Show WSCG config
print("\nWhole-Stage CodeGen enabled:",
      spark.conf.get("spark.sql.codegen.wholeStage"))
print("Max fields for WSCG     :",
      spark.conf.get("spark.sql.codegen.maxFields", "100"))
print("Codegen fallback enabled:",
      spark.conf.get("spark.sql.codegen.fallback", "true"))

### Problem Statement
Compare a Python UDF pipeline vs a native Spark expression pipeline. Observe how the UDF breaks WholeStageCodegen and forces row-by-row serialization across the JVM/Python boundary via Py4J and Apache Arrow.

In [ ]:
# ── Poorly Optimized: Python UDF breaks Tungsten ─────────────────────────────
from pyspark.sql.functions import udf

@udf(returnType=StringType())
def normalize_name_udf(name, license_type):
    """Python UDF — crosses JVM/Python boundary for EVERY row.
    Serialization path: UnsafeRow -> Java object -> Pickle/Arrow -> Python object
    De-serialization:   Python result -> Pickle/Arrow -> Java object -> UnsafeRow
    Cost: ~100-1000x slower than equivalent native Spark expression.
    GC impact: Creates new Python and Java objects for every row.
    """
    if name and license_type:
        return f"{name.strip().upper()}_{license_type.replace(' ', '_')}"
    return None

@udf(returnType=IntegerType())
def vehicle_age_udf(year):
    """Second UDF — another JVM/Python boundary crossing per row."""
    if year:
        return 2024 - year
    return None

df_udf = (
    cabs_t2
    .filter(F.col("Active") == "YES")
    .withColumn("NormalizedName", normalize_name_udf(F.col("Name"), F.col("LicenseType")))
    .withColumn("VehicleAge",     vehicle_age_udf(F.col("VehicleYear")))
    .select("CabNumber", "NormalizedName", "VehicleAge")
)

print("=" * 70)
print("UDF Pipeline — explain(extended=True):")
print("=" * 70)
df_udf.explain(extended=True)
# Look for: NO *(N) prefix = WholeStageCodegen DISABLED by UDF
# Look for: BatchEvalPython or ArrowEvalPython in physical plan

### Interview Questions — Topic 2: Tungsten Engine

1. **UnsafeRow binary format:** Describe the exact memory layout of an `UnsafeRow` for a row with schema `(INT, STRING, DOUBLE, STRING)`. Where does the null bitmap live? How is the variable-length string data referenced from the fixed-length section? What is the total memory footprint?

2. **sun.misc.Unsafe justification:** What is `sun.misc.Unsafe` and why does Spark use it instead of `java.nio.ByteBuffer`? What are the three specific operations in Spark's codebase that REQUIRE Unsafe semantics and cannot be implemented with standard Java APIs?

3. **WSCG compilation pipeline:** WholeStageCodegen generates Java source code and compiles it via Janino at query runtime. Walk through what happens when the generated code exceeds Janino's 64KB method size limit. What is `spark.sql.codegen.hugeMethodLimit` and what is the fallback behavior?

4. **UDF boundary costs:** Quantify the serialization cost of a Python UDF. What is the exact data path from an `UnsafeRow` in executor JVM memory to a Python dictionary, through Arrow IPC format? What does `spark.sql.execution.arrow.pyspark.enabled` change about this path?

5. **Vectorized execution:** What is Apache Arrow's columnar format and how does `pandas_udf` (Vectorized UDF) use it to reduce serialization overhead vs row-at-a-time UDFs? What are the limitations of pandas_udf that prevent full Tungsten acceleration?

6. **Memory modes:** Spark has three memory modes: on-heap execution, off-heap execution, and on-heap storage. Explain the purpose of each. When would you enable `spark.memory.offHeap.enabled`? What GC behavior change do you expect?

7. **WSCG operator support:** Not all Spark operators support CodegenSupport. Name five operators that CANNOT participate in WholeStageCodegen and explain the architectural reason each one breaks the tight-loop compilation model.

8. **Tungsten sort:** Describe Tungsten's sort algorithm. How does it differ from a standard comparison-based sort? What is a "prefix key" in the context of `RadixSort`? When does Spark choose RadixSort vs TimSort, and what data characteristics trigger each?

### Expected Logical Plan Discussion

**UDF Pipeline Logical Plan:**
```
Project [CabNumber, NormalizedName, VehicleAge]
+- Project [CabNumber, NormalizedName, vehicle_age_udf(VehicleYear) AS VehicleAge]
   +- Project [CabNumber, Name, VehicleYear, normalize_name_udf(Name, LicenseType) AS NormalizedName]
      +- Filter (Active = YES)
         +- Relation cabs_t2
```

**Key issue in the logical plan:** Catalyst cannot look inside a UDF. The UDF appears as an opaque `ScalaUDF` or `PythonUDF` node. Catalyst cannot:
- Constant-fold values through a UDF
- Push predicates through a UDF
- Eliminate a UDF call if its output is not used
- Combine two UDF calls into one (even if the developer intended them to be fused)

**Native Pipeline Logical Plan:**
```
Project [CabNumber, NormalizedName, VehicleAge]
+- Project [CabNumber,
            concat(upper(trim(Name)), _, replace(LicenseType, ' ', '_')) AS NormalizedName,
            (2024 - VehicleYear) AS VehicleAge]
   +- Filter (Active = YES)
      +- Relation cabs_t2
```
Catalyst can now: apply ConstantFolding to `(2024 - VehicleYear)`, ColumnPrune unused columns, and fuse the filter + projection into a single WholeStageCodegen stage.

### Expected Physical Plan Discussion

**UDF Pipeline Physical Plan** (what you will see in explain()):
```
ArrowEvalPython [vehicle_age_udf(VehicleYear)]        ← Python boundary
+- ArrowEvalPython [normalize_name_udf(Name, LicenseType)]  ← Python boundary
   +- *(1) Filter (Active = YES)                    ← WSCG stops here
      +- *(1) FileScan csv [...]
```

**Two separate ArrowEvalPython nodes** = two round trips per row to Python.
- Row 1: Java→Arrow batch→Python (normalize)→Arrow→Java ... Python (age)→Arrow→Java
- The `*(1)` WSCG stage covers only the scan and filter. Everything after the first UDF is interpreted.

**Native Pipeline Physical Plan:**
```
*(1) Project [CabNumber, concat(upper(trim(Name)), ...) AS NormalizedName, (2024-VehicleYear) AS VehicleAge]
+- *(1) Filter (Active = YES)
   +- *(1) FileScan csv [CabNumber, Name, LicenseType, Active, VehicleYear]
```
**Single `*(1)` stage** — scan + filter + projection all compiled into one Java method. Zero Python boundary crossings. All operations on `UnsafeRow` binary data via pointer arithmetic.

### Spark UI Investigation Guide — Tungsten / WSCG Debugging

**Stages Tab:** (`/stages`)
- **GC Time column:** Shows cumulative GC time per stage. Healthy: `<10%` of task time. Bad: `>20%`. In our UDF scenario, expect 30-50% GC time because each Python boundary crossing creates thousands of short-lived Java objects.
- **Task Deserialization Time:** Time to deserialize the task closure (your UDF function object). If this is high (`>100ms`), your UDF closure is capturing large objects (a common Python closure bug).

**Task Metrics Detail (click a stage → Aggregated Metrics by Executor):**
- `Python Evaluation Time` — available when Arrow/UDF is in use. This is the ACTUAL python execution time, separate from serialization.
- `Input Rows` vs `Output Rows` — if a filter is working, output should be much less than input.

**SQL/DataFrame Tab — DAG:**
- UDF pipeline: you will see `ArrowEvalPython` or `BatchEvalPython` boxes in the DAG. Each one is a boundary.
- Native pipeline: all boxes show `WholeStageCodegen (1)` — one stage, no boundary.
- **Tip:** Count the `WholeStageCodegen` boundaries. Each new WSCG stage after a boundary = operators that are NOT fused = interpreted execution in that boundary gap.

**Environment Tab → Spark Properties:**
- Check `spark.sql.codegen.wholeStage` = `true`
- Check `spark.sql.execution.arrow.pyspark.enabled` = `true` (enables Arrow path for UDFs)
- Check `spark.sql.codegen.fallback` = `true` (prevents hard failures when WSCG fails)

**War Story:** In a production pipeline processing 2TB/day, a single `@udf` on a 200-char string field caused 4× slowdown and 45% GC rate. Rewriting the UDF as `F.concat(F.upper(F.trim(col)), F.lit("_"), F.regexp_replace(col2, " ", "_"))` reduced runtime from 48 minutes to 11 minutes and GC dropped to 3%.

In [ ]:
# ── Topic 2: Optimization Exercise ──────────────────────────────────────────
# Replace Python UDFs with native Spark SQL functions

df_native = (
    cabs_t2
    .filter(F.col("Active") == "YES")
    # Replace normalize_name_udf with native functions:
    .withColumn("NormalizedName",
        F.concat(
            F.upper(F.trim(F.col("Name"))),
            F.lit("_"),
            F.regexp_replace(F.col("LicenseType"), " ", "_")
        )
    )
    # Replace vehicle_age_udf with arithmetic:
    .withColumn("VehicleAge", F.lit(2024) - F.col("VehicleYear"))
    .select("CabNumber", "NormalizedName", "VehicleAge")
)

print("Native Pipeline — Physical Plan (look for *(1) on ALL operators):")
df_native.explain()

# Pandas UDF (vectorized) — better than row UDF, still has boundary cost
from pyspark.sql.functions import pandas_udf
import pandas as pd

@pandas_udf(StringType())
def normalize_name_pandas_udf(name: pd.Series, license_type: pd.Series) -> pd.Series:
    """
    Pandas UDF (Vectorized): processes an entire Arrow batch at once.
    Serialization: JVM Arrow IPC → Python pandas Series → Arrow IPC → JVM
    Cost: ~10x cheaper than row-at-a-time UDF, ~10x more expensive than native.
    Use ONLY when logic cannot be expressed in native Spark SQL functions.
    """
    return (name.str.strip().str.upper() + "_" +
            license_type.str.replace(" ", "_", regex=False))

df_pandas_udf = (
    cabs_t2
    .filter(F.col("Active") == "YES")
    .withColumn("NormalizedName",
        normalize_name_pandas_udf(F.col("Name"), F.col("LicenseType")))
    .withColumn("VehicleAge", F.lit(2024) - F.col("VehicleYear"))
    .select("CabNumber", "NormalizedName", "VehicleAge")
)

print("\nPandas UDF Pipeline — Physical Plan (look for ArrowEvalPython):")
df_pandas_udf.explain()

In [ ]:
# ── Topic 2: Performance Benchmarking ────────────────────────────────────────
import time

results = {}

for label, df in [
    ("Row UDF (worst)",       df_udf),
    ("Pandas UDF (middle)",   df_pandas_udf),
    ("Native Spark (best)",   df_native),
]:
    # Run twice — first warms JIT/Janino cache
    df.count()   # warm-up
    start = time.time()
    n = df.count()
    elapsed = time.time() - start
    results[label] = elapsed
    print(f"{label:<30} {n:>8,} rows  {elapsed:>6.2f}s")

print("\n--- Relative Performance ---")
baseline = results["Native Spark (best)"]
for label, t in results.items():
    print(f"{label:<30} {t/baseline:>5.2f}x slower than native")

print("""
Expected results (approximate):
  Row UDF:      5-20x slower (depends on Python startup overhead)
  Pandas UDF:   1.5-3x slower (Arrow batch amortizes boundary cost)
  Native:       1.0x baseline (WSCG tight loop, no Python boundary)

At 10TB scale:
  Row UDF:      48 hours
  Pandas UDF:   8 hours
  Native:       3 hours
""")

### Production Best Practices — Tungsten Engine

1. **Zero-tolerance policy for Python UDFs in hot paths.** Any `@udf` in a loop over >1M rows is a performance bug. Enforce via code review checklist or static analysis lint rule. Provide a Spark native function reference to all developers.

2. **Use `pandas_udf` when business logic requires Python libraries** (e.g., `scipy`, `sklearn`, custom regex). The Arrow batch path reduces boundary-crossing cost by 10-50× vs row UDFs. Always specify `spark.sql.execution.arrow.pyspark.enabled=true`.

3. **Enable Arrow for `toPandas()` and `createDataFrame(pandas_df)`.** Without Arrow, these operations serialize each row as a Java object. With Arrow: `spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")` reduces collection time by 5-10× on wide DataFrames.

4. **Monitor `spark.sql.codegen.wholeStage` in production.** WSCG can be silently disabled if: a plan has >100 fields (`spark.sql.codegen.maxFields`), an operator doesn't implement `CodegenSupport`, or Janino compilation fails. Check the physical plan for missing `*(N)` prefixes.

5. **Off-heap memory (`spark.memory.offHeap.enabled=true`) reduces GC pauses** for shuffle-heavy workloads. Shuffle data is stored in off-heap `UnsafeRow` format. Set `spark.memory.offHeap.size` to 50-80% of executor memory. The GC has less heap to scan between collections.

6. **UnsafeRow is not thread-safe.** If you use multi-threaded UDFs or `foreachPartition` with thread pools, UnsafeRow references will be invalid after the iterator moves. Always call `row.copy()` before storing a reference across iterator boundaries.

7. **`spark.sql.codegen.fallback=true` (default) silently degrades to interpreted mode.** Set it to `false` in CI/test environments to catch operators that break WSCG early. Never set to `false` in production — it will cause hard failures on plan changes.

8. **Tungsten's sort is optimized for fixed-length types.** If you `orderBy` on a string column in a large shuffle, consider hashing it to a long first. Radix sort on longs is O(n) vs O(n log n) comparison sort on strings.

9. **`spark.sql.codegen.maxFields=100` (default).** DataFrames with >100 columns silently fall back to interpreted mode. In wide-table scenarios (Hive schemas with 200+ columns), either prune columns before the expensive operation or increase this limit.

10. **Profile with `spark.executor.extraJavaOptions=-XX:+PrintGCDetails -XX:+PrintGCTimeStamps`.** Route GC logs to a file and parse with GCViewer or Gceasy.io. High `Old Gen GC` frequency indicates long-lived objects accumulating — often from caching or large UDF closures.

### Follow-up Questions — Topic 2

- How does Spark's `UnsafeKVExternalSorter` use off-heap memory during an external sort? What triggers a "spill" and what is written to disk during a spill?
- `spark.sql.execution.arrow.maxRecordsPerBatch` controls Arrow batch size for pandas UDFs. What is the tradeoff between a large batch (e.g., 100,000 rows) and a small batch (e.g., 1,000 rows)?
- Explain `CodegenFallback` — the trait that operators implement when they cannot generate code. What does the fallback interpreter do that is slower than WSCG?
- Databricks Runtime uses Photon, a vectorized C++ execution engine. How does Photon differ from Spark's Tungsten WSCG approach? What operators does Photon accelerate that WSCG does not?

In [ ]:
# ── Topic 2: Expert Solution — WSCG Audit + UDF Migration Framework ──────────

def tungsten_audit(df, label="DataFrame"):
    """
    Audit a DataFrame pipeline for Tungsten compliance.
    Returns a compliance score (0-100) and recommendations.
    Production use: run this in CI before merging ETL PRs.
    """
    physical = str(df.queryExecution.executedPlan)
    lines = physical.split("\n")

    # Score components
    score = 100
    issues = []

    # UDF checks
    python_udf_count = physical.count("ArrowEvalPython") + physical.count("BatchEvalPython")
    if python_udf_count > 0:
        penalty = python_udf_count * 15
        score -= penalty
        issues.append(f"Python UDF boundaries: {python_udf_count} "
                      f"(-{penalty} pts) → rewrite with native functions")

    # WSCG coverage
    wscg_lines = [l for l in lines if l.strip().startswith("*(")]
    non_wscg_ops = [l for l in lines if l.strip().startswith("+- ") and
                    not any(l2.strip().startswith("*(") for l2 in lines[:lines.index(l)])]
    if python_udf_count == 0 and not wscg_lines:
        score -= 30
        issues.append("No WholeStageCodegen stages detected (-30 pts) → check codegen.maxFields")

    # Exchange (shuffle) count
    exchanges = physical.count("Exchange")
    if exchanges > 3:
        penalty = (exchanges - 3) * 5
        score -= penalty
        issues.append(f"High shuffle count: {exchanges} exchanges "
                      f"(-{penalty} pts) → reduce joins or use broadcast")

    # ObjectHashAggregate check
    if "ObjectHashAggregate" in physical:
        score -= 20
        issues.append("ObjectHashAggregate detected (-20 pts) → avoid complex type UDAFs")

    score = max(0, score)
    grade = "A" if score >= 90 else "B" if score >= 75 else "C" if score >= 60 else "F"

    print(f"\n{'='*65}")
    print(f"Tungsten Compliance Audit: {label}")
    print(f"{'='*65}")
    print(f"  Score: {score}/100  Grade: {grade}")
    print(f"  WSCG stages       : {len(wscg_lines)}")
    print(f"  Python UDF bounds : {python_udf_count}")
    print(f"  Exchange (shuffle): {exchanges}")
    if issues:
        print("\n  Issues:")
        for issue in issues:
            print(f"    - {issue}")
    else:
        print("  No issues — fully Tungsten-compliant pipeline.")

tungsten_audit(df_udf,        "Row UDF Pipeline")
tungsten_audit(df_pandas_udf, "Pandas UDF Pipeline")
tungsten_audit(df_native,     "Native Spark Pipeline")

## Topic 3 — Cost-Based Optimizer (CBO)

### Scenario
A data warehouse team runs a query joining four cab-related tables: cabs, licenses, vehicle types, and address regions. The query takes 8 minutes. After enabling CBO and running ANALYZE TABLE, the same query takes 90 seconds. You must explain why and demonstrate the mechanism.

### CBO Architecture
```
Rule-Based Optimizer (RBO)        Cost-Based Optimizer (CBO)
───────────────────────────────   ────────────────────────────────────────
Applies fixed rules always        Uses statistics to make decisions
Ignores data distribution         Models row counts, NDV, byte sizes
Join order = code order           Join order = DP algorithm on cost tree
No selectivity model              Selectivity = f(NDV, histograms)
Fast plan generation              Slower planning, much better plans

CBO Statistics (collected by ANALYZE TABLE):
  - rowCount          : total rows in table
  - sizeInBytes       : total byte size
  - Per-column stats:
      distinctCount   : NDV (number of distinct values)
      min / max       : range boundaries
      nullCount       : null rows
      avgLen          : average byte length
      maxLen          : max byte length
      histogram       : equi-height histogram (with spark.sql.statistics.histogram.enabled)

CBO Join Reordering (spark.sql.cbo.joinReorder.enabled):
  - Dynamic Programming over all join permutations
  - dp.threshold (default=12): max tables before DP is too expensive
  - Cost = outputRows x outputBytes (lower = better)
  - Filters selectivity is estimated using column stats
```

In [ ]:
# ── Topic 3: Data Setup — four tables to join ────────────────────────────────
spark.conf.set("spark.sql.cbo.enabled", "true")
spark.conf.set("spark.sql.cbo.joinReorder.enabled", "true")
spark.conf.set("spark.sql.statistics.histogram.enabled", "true")

# Table 1: cabs (large ~8970 rows)
cabs_raw.createOrReplaceTempView("t_cabs")

# Table 2: license_types (tiny 3 rows)
license_types = spark.createDataFrame([
    ("OWNER MUST DRIVE",   "O",  True),
    ("NAMED DRIVER",       "N",  True),
    ("OWNER NAMED DRIVER", "ON", False),
], ["LicenseType", "Code", "IsActive"])
license_types.createOrReplaceTempView("t_license_types")

# Table 3: vehicle_types (small 10 rows)
vehicle_types = spark.createDataFrame([
    ("SEDAN","S",4),("SUV","U",5),("VAN","V",7),("TRUCK","T",2),
    ("HYBRID","H",4),("ELECTRIC","E",4),("TAXI","X",4),
    ("LIMO","L",6),("BUS","B",20),("OTHER","O",4)
], ["VehicleType","TypeCode","MaxPassengers"])
vehicle_types.createOrReplaceTempView("t_vehicle_types")

# Table 4: borough_map (medium 50 rows)
boroughs = [("BROOKLYN","BK",2),("MANHATTAN","MN",1),
            ("QUEENS","QN",4),("BRONX","BX",3),("STATEN ISLAND","SI",5)]
borough_rows = [(f"ZONE_{i:03d}", boroughs[i%5][0], boroughs[i%5][1], boroughs[i%5][2])
                for i in range(50)]
borough_map = spark.createDataFrame(borough_rows,
    ["ZoneCode","Borough","BoroughCode","DistrictNum"])
borough_map.createOrReplaceTempView("t_borough_map")

print("Tables registered:")
for t in ["t_cabs","t_license_types","t_vehicle_types","t_borough_map"]:
    n = spark.table(t).count()
    print(f"  {t:<22} {n:>6,} rows")

### Problem Statement
Run ANALYZE TABLE on all four tables. Compare the query plan before and after — specifically, watch how join ordering changes when CBO has accurate row counts and NDV statistics.

In [ ]:
# ── Without CBO: RBO picks join order based on code order ────────────────────
spark.conf.set("spark.sql.cbo.enabled", "false")
spark.conf.set("spark.sql.cbo.joinReorder.enabled", "false")

df_no_cbo = spark.sql("""
    SELECT c.CabNumber, c.Name, lt.Code, vt.TypeCode, bm.Borough
    FROM   t_cabs           c
    JOIN   t_license_types  lt ON c.LicenseType = lt.LicenseType
    JOIN   t_vehicle_types  vt ON c.VehicleType = vt.VehicleType
    JOIN   t_borough_map    bm ON bm.BoroughCode = 'MN'
    WHERE  c.Active = 'YES'
    AND    lt.IsActive = true
""")

print("WITHOUT CBO — Physical Plan (join order = code order):")
df_no_cbo.explain()
# Expected: cabs is the root (largest first), then smaller tables join on top
# Suboptimal: large intermediate results at every step

### Interview Questions — Topic 3: Cost-Based Optimizer

1. **Default cardinality estimation:** How does CBO estimate the output cardinality of a Filter node when there are NO column statistics available? What is the default selectivity assumption for equality predicates, range predicates, and LIKE predicates? Why are these defaults often wrong?

2. **NDV propagation through joins:** If table A has 1,000 rows with NDV(key)=100, and table B has 10,000 rows with NDV(key)=500, what does CBO estimate as the output cardinality of `A JOIN B ON A.key = B.key`? Walk through the exact formula.

3. **Histogram types:** Spark CBO supports equi-height histograms. Explain what equi-height means vs equi-width. When does an equi-height histogram give a significantly more accurate selectivity estimate than a min/max range?

4. **ANALYZE limitations:** ANALYZE TABLE computes statistics at collection time. In a streaming or frequently-updated table, statistics become stale. What strategies do you use to keep CBO statistics fresh without running full ANALYZE on every write?

5. **Join reorder DP algorithm:** CBO's join reordering uses dynamic programming. Describe the DP state: what does each subplan in the DP table represent? What is the cost function? How does `spark.sql.cbo.joinReorder.dp.threshold=12` affect this?

6. **CBO vs AQE:** AQE's `OptimizeSkewedJoin` and `CoalesceShufflePartitions` also use runtime statistics. How do AQE's runtime statistics differ from CBO's compile-time statistics? Can they contradict each other? What happens when they do?

7. **Filter on joined column:** After a join `A JOIN B ON A.id = B.id WHERE A.val > 100`, CBO needs to estimate the selectivity of the filter. Explain how CBO uses column statistics from BOTH sides of the join condition to estimate the result cardinality.

8. **Multi-column statistics:** Spark CBO does not natively support multi-column correlation statistics. Explain why this is a problem for queries with correlated predicates like `WHERE city = 'NYC' AND state = 'NY'`. What workarounds exist?

### Expected Logical Plan Discussion

**Without CBO** — Logical plan reflects code order. The `Join` nodes are ordered exactly as written in SQL:
```
Project [CabNumber, Name, Code, TypeCode, Borough]
+- Filter (Active=YES AND IsActive=true)
   +- Join Inner (LicenseType=LicenseType)       ← outermost join = last in code
      :- Join Inner (VehicleType=VehicleType)
      :  :- Join Inner (BoroughCode=MN)
      :  :  :- Relation t_cabs                   ← LARGEST at the root
      :  :  +- Relation t_borough_map
      :  +- Relation t_vehicle_types
      +- Relation t_license_types
```
The optimizer applies `ReorderJoin` but without statistics it cannot determine which ordering is cheapest. It falls back to a heuristic that tends to keep the original order.

**With CBO** — After `ANALYZE TABLE`, each relation node is annotated with `Statistics(sizeInBytes=..., rowCount=...)`. The `CostBasedJoinReorder` rule applies DP:
- DP considers all 4! = 24 possible left-deep join trees
- Each candidate is costed: `cost = outputRows × outputSizeBytes`
- `t_license_types` filtered by `IsActive=true` → estimated 2 rows. Joining first with `t_cabs` filtered by `Active=YES` → ~6,000 rows output
- This 6,000-row intermediate is then joined to `t_vehicle_types` (10 rows) → ~6,000 rows (since VehicleType is mostly NULL in cabs)
- Finally joined to `t_borough_map` (50 rows) filtered to `BoroughCode=MN` → 10 rows

The CBO-chosen plan builds up from the smallest tables, keeping intermediate results compact at every step.

### Expected Physical Plan Discussion

**Without CBO — Physical Plan:**
- All four tables are small enough that Spark broadcasts them all (BroadcastHashJoin for each)
- BUT the join TREE is left-deep starting with `t_cabs` as the probe side of every join
- `t_cabs` (8,970 rows) is the streaming side, never filtered before the join chain begins
- Each subsequent join probes the 8,970-row stream against a broadcast hash table

**With CBO — Physical Plan:**
- CBO may reorder to start with `t_license_types` (2 rows after filter) as the build side
- `t_cabs` filtered by `Active=YES` becomes the probe side for the first join
- The filter on `Active` now runs before the join, reducing the probe side to ~6,000 rows
- Statistics annotations are visible in the plan: `Statistics(sizeInBytes=..., rowCount=...)`

**Key metric to watch in the plan:**
```
Filter (Active = YES), Statistics(sizeInBytes=480.0 KB, rowCount=6700)
```
vs
```
Relation t_cabs, Statistics(sizeInBytes=1.2 MB, rowCount=8970)
```
The filter reduces estimated rows from 8,970 to 6,700. CBO uses this to place the filter BEFORE it estimates join output sizes, producing a more accurate cost model for the entire join tree.

### Spark UI Investigation Guide — CBO Validation

**SQL Tab → Query Details → Physical Plan (click "Details"):**
- Look for `Statistics(sizeInBytes=..., rowCount=...)` annotations on each plan node
- Without ANALYZE: `sizeInBytes=8.0 EiB` (default unknown = Long.MaxValue). CBO is guessing
- With ANALYZE: `sizeInBytes=1.2 MB, rowCount=8970`. CBO has real data

**Reading the statistics annotations:**
```
+- Filter (Active = YES), Statistics(sizeInBytes=512.0 KB, rowCount=6700)
   +- Relation t_cabs, Statistics(sizeInBytes=1.2 MB, rowCount=8970)
```
Filter selectivity = 6700/8970 = 0.747. CBO estimated 74.7% of rows pass `Active = YES`.
If this selectivity looks wrong, your histogram stats may be stale or the column has extreme skew.

**Stages Tab — spot the CBO effect:**
- Check `Input Size / Records` on each stage
- Without CBO: Stage 1 processes all 8,970 cabs rows as the probe side
- With CBO: Stage 1 processes only 2 license_type rows as the build side
- The stage with the MOST input records is where CBO has the biggest impact

**Environment Tab → Spark Properties — verify CBO is active:**
- `spark.sql.cbo.enabled` = `true`
- `spark.sql.cbo.joinReorder.enabled` = `true`
- `spark.sql.statistics.histogram.enabled` = `true`

**War Story:** A 6-table reporting query at a financial firm ran 22 minutes nightly. After `ANALYZE TABLE ... COMPUTE STATISTICS FOR ALL COLUMNS` on all 6 tables, CBO reordered the joins and the query ran in 4 minutes — zero code changes. Annual savings: ~$180K in cluster compute costs.

In [ ]:
# ── Topic 3: Optimization — Enable CBO and Run ANALYZE ───────────────────────
spark.conf.set("spark.sql.cbo.enabled", "true")
spark.conf.set("spark.sql.cbo.joinReorder.enabled", "true")

for table in ["t_cabs","t_license_types","t_vehicle_types","t_borough_map"]:
    spark.sql(f"ANALYZE TABLE {table} COMPUTE STATISTICS FOR ALL COLUMNS")
    print(f"Analyzed: {table}")

# Re-run same query WITH CBO
df_with_cbo = spark.sql("""
    SELECT c.CabNumber, c.Name, lt.Code, vt.TypeCode, bm.Borough
    FROM   t_cabs           c
    JOIN   t_license_types  lt ON c.LicenseType = lt.LicenseType
    JOIN   t_vehicle_types  vt ON c.VehicleType = vt.VehicleType
    JOIN   t_borough_map    bm ON bm.BoroughCode = 'MN'
    WHERE  c.Active = 'YES'
    AND    lt.IsActive = true
""")

print("\nWITH CBO — Physical Plan (observe changed join order and statistics annotations):")
df_with_cbo.explain()

In [ ]:
# ── Topic 3: Benchmark ────────────────────────────────────────────────────────
import time

spark.conf.set("spark.sql.cbo.enabled", "false")
spark.conf.set("spark.sql.cbo.joinReorder.enabled", "false")
df_no_cbo.count()
t0 = time.time(); n1 = df_no_cbo.count(); t_no_cbo = time.time() - t0

spark.conf.set("spark.sql.cbo.enabled", "true")
spark.conf.set("spark.sql.cbo.joinReorder.enabled", "true")
df_with_cbo.count()
t0 = time.time(); n2 = df_with_cbo.count(); t_cbo = time.time() - t0

print(f"Without CBO : {n1:,} rows in {t_no_cbo:.2f}s")
print(f"With CBO    : {n2:,} rows in {t_cbo:.2f}s")
print(f"Speedup     : {t_no_cbo/max(t_cbo,0.001):.2f}x")
print()
print("Note: On this small dataset (~9K rows) both plans broadcast all tables.")
print("At scale (1TB+ per table), CBO join reordering yields 3-10x speedup")
print("by avoiding large intermediate shuffle datasets.")

### Production Best Practices — Cost-Based Optimizer

1. **Always enable CBO in production:** `spark.sql.cbo.enabled=true`, `spark.sql.cbo.joinReorder.enabled=true`. Add to cluster-level `spark-defaults.conf` — not per-job config — so every query benefits without developer action.

2. **Run ANALYZE after significant data changes.** Integrate `ANALYZE TABLE ... COMPUTE STATISTICS FOR ALL COLUMNS` as the final step of every major ETL write. Stale stats are worse than no stats — CBO makes confidently wrong decisions based on old row counts.

3. **Use column-level statistics selectively on wide tables.** `FOR ALL COLUMNS` is expensive on 200+ column schemas. Identify join keys and high-selectivity filter columns: `COMPUTE STATISTICS FOR COLUMNS col1, col2, col3`. 10-50x faster than full column stats.

4. **Enable histograms for skewed columns.** `spark.sql.statistics.histogram.enabled=true`. For columns like `borough`, `license_type`, `vehicle_year` where values cluster unevenly, histograms give dramatically better selectivity estimates than min/max range alone.

5. **`spark.sql.cbo.joinReorder.dp.threshold=12` is the join reorder limit.** For queries joining >12 tables (common in BI/reporting), CBO falls back to a greedy algorithm. For joins beyond this threshold, provide explicit ordering hints or restructure the query into CTEs.

6. **Delta Lake auto-collects statistics on write.** Every `INSERT`/`MERGE` on a Delta table updates statistics for the first 32 columns (`dataSkippingNumIndexedCols`). CBO always has fresh stats for Delta — a major advantage over raw Parquet/ORC.

7. **Combine CBO with broadcast hints for defense-in-depth.** CBO might correctly identify a small table but fail to broadcast it if the computed size is just above `autoBroadcastJoinThreshold`. Always add explicit `F.broadcast()` for tables you know are small.

8. **Verify CBO is using statistics, not guessing.** `df.explain(True)` shows `Statistics(sizeInBytes=..., rowCount=...)` per node. `sizeInBytes=8.0 EiB` = CBO has no data = guessing. Run ANALYZE or add hints.

9. **AQE supplements CBO for runtime skew.** Even with perfect CBO stats, data skew at runtime cannot be predicted at planning time. Enable `spark.sql.adaptive.enabled=true` alongside CBO — they are complementary, not alternatives.

10. **A/B test CBO impact before enabling cluster-wide.** CBO can occasionally make plans WORSE if statistics are stale or query patterns are unusual. Test: run 20 representative queries with CBO on/off, compare runtimes and plans. Keep a plan audit trail in CI.

### Follow-up Questions — Topic 3

- Iceberg and Delta Lake maintain their own statistics in table metadata. How do these integrate with Spark's CBO? Does Spark read Iceberg's `puffin` statistics files, or does it require a separate ANALYZE step?
- In a query with 3 joins and 2 filters, CBO correctly reorders the joins but fails to push a filter through a complex expression. How would you detect this gap and what is the workaround?
- What is `spark.sql.cbo.planStats.enabled`? How does it differ from `spark.sql.cbo.enabled`? Can you use plan-level statistics without enabling full CBO join reordering?
- How does `EXPLAIN COST` (available in some Spark distributions) differ from `EXPLAIN` in terms of what statistics are shown?

In [ ]:
# ── Topic 3: Expert Solution — CBO Statistics Health Check ───────────────────

def cbo_health_check(spark, table_names):
    """
    Audits CBO statistics across a list of temp views/tables.
    Returns tables needing re-analysis.
    Production use: run before any query that uses CBO join reordering.
    """
    needs_analyze = []
    print(f"{'Table':<25} {'Rows':>10} {'HasStats':>10} {'Note'}")
    print("-" * 65)

    for table in table_names:
        try:
            df_tmp = spark.table(table)
            # Access Java stats object via queryExecution
            j_stats = df_tmp.queryExecution.analyzed.stats()
            size_bytes = j_stats.sizeInBytes()
            # Long.MaxValue / 8 EiB = no statistics collected
            EIB_8 = 9_223_372_036_854_775_807  # Long.MaxValue
            no_stats = (size_bytes >= EIB_8 // 2)

            row_count = df_tmp.count()
            status = "MISSING" if no_stats else "OK"
            note = "Run ANALYZE" if no_stats else f"size={size_bytes:,}B"
            print(f"  {table:<23} {row_count:>10,} {status:>10}  {note}")

            if no_stats:
                needs_analyze.append(table)
        except Exception as e:
            print(f"  {table:<23} {'ERROR':>10}  {str(e)[:40]}")
            needs_analyze.append(table)

    print()
    if needs_analyze:
        print("ANALYZE commands needed:")
        for t in needs_analyze:
            print(f"  spark.sql('ANALYZE TABLE {t} COMPUTE STATISTICS FOR ALL COLUMNS')")
    else:
        print("All tables have CBO statistics. Join reordering is fully informed.")
    return needs_analyze

# Run before CBO-dependent queries
_ = cbo_health_check(spark, ["t_cabs","t_license_types","t_vehicle_types","t_borough_map"])

## Topic 4 — Join Strategy Selection

### Scenario
Your team has four different join patterns in a single pipeline: a tiny lookup table (3 rows), a medium dimension (50K rows), a large fact table (10M rows), and a self-join for deduplication. Each requires a different join strategy. In the interview you must explain EXACTLY when Spark picks each algorithm, the memory requirements, and how to override Spark's choice.

### All 5 Join Strategies

| Strategy | When Chosen | Shuffles? | Memory Req | Best For |
|---|---|---|---|---|
| **BroadcastHashJoin** | One side < `autoBroadcastJoinThreshold` (10MB default) | 0 (probe side) | Build side × all executors | Small lookup tables |
| **ShuffleHashJoin** | Both shuffled; build partition fits in memory | 2 | Build partition in RAM | Medium tables, avoid sort cost |
| **SortMergeJoin** | Default for large-large joins with sortable key | 2 | O(1) streaming merge | Large-large equi-joins |
| **BroadcastNestedLoopJoin** | No equi-join condition; small build side | 0 (build broadcast) | Build side × all executors | Non-equi joins (range, LIKE) |
| **CartesianProduct** | Explicit `CROSS JOIN`, no condition | 0 | Entire right side in memory | Explicit cross products only |

### JoinSelection Decision Tree (SparkPlanner Strategy)
```
JoinSelection.apply(join)
  1. broadcast hint present?              → BroadcastHashJoin (or BroadcastNLJ for non-equi)
  2. Left/right size < autoBroadcastJoinThreshold AND stats known?
                                          → BroadcastHashJoin
  3. shuffle_hash hint?                   → ShuffleHashJoin
  4. merge hint?                          → SortMergeJoin
  5. spark.sql.join.preferSortMergeJoin=true (default) AND key sortable?
                                          → SortMergeJoin
  6. Build side fits in one partition's memory budget?
                                          → ShuffleHashJoin
  7. Fallback                             → SortMergeJoin
  AQE override: after shuffle, if actual size < autoBroadcastJoinThreshold
                                          → convert SMJ → BroadcastHashJoin
```

In [ ]:
# ── Topic 4: Data Setup ───────────────────────────────────────────────────────
# Tiny: 3-row lookup (broadcast candidate)
tiny_lookup = spark.createDataFrame([
    ("OWNER MUST DRIVE","O"), ("NAMED DRIVER","N"), ("OWNER NAMED DRIVER","ON")
], ["LicenseType","Code"])

# Left side: cabs_raw (~8,970 rows)
cabs_left = cabs_raw.select("CabNumber","LicenseType","VehicleYear","Active")

# Right side: 5x union (~44,850 rows) for medium join
cabs_right = cabs_raw
for _ in range(4):
    cabs_right = cabs_right.union(cabs_raw)
cabs_right = cabs_right.select("CabNumber","Name","LicenseType","VehicleYear")

print("Join strategy demo tables:")
print(f"  tiny_lookup : {tiny_lookup.count():>8,} rows  (broadcast candidate)")
print(f"  cabs_left   : {cabs_left.count():>8,} rows  (probe side)")
print(f"  cabs_right  : {cabs_right.count():>8,} rows  (larger side)")
print(f"\nautoBroadcastJoinThreshold: "
      f"{int(spark.conf.get('spark.sql.autoBroadcastJoinThreshold')):,} bytes "
      f"({int(spark.conf.get('spark.sql.autoBroadcastJoinThreshold'))//1024//1024} MB)")

### Problem Statement
Demonstrate all five join strategies using hints. Show the physical plan for each. Explain why each was selected and its memory/performance tradeoffs.

In [ ]:
# ── Poorly Optimized: Default join, no hints, broadcast disabled ──────────────
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # force no auto-broadcast

df_default_join = cabs_left.join(cabs_right, "LicenseType", "inner")

print("DEFAULT join (broadcast disabled) — falls through to SortMergeJoin:")
df_default_join.explain()
# SortMergeJoin: 2 full shuffles + 2 sorts
# Both sides exchange on hashpartitioning(LicenseType, 200)
# Then Sort on LicenseType ASC, then merge scan

### Interview Questions — Topic 4: Join Strategy Selection

1. **ShuffleHashJoin vs SortMergeJoin:** `spark.sql.join.preferSortMergeJoin=true` by default. Under what exact conditions would Spark choose ShuffleHashJoin OVER SortMergeJoin even with this flag set? What is the memory requirement for the build side of a ShuffleHashJoin per partition and how does it relate to `spark.sql.shuffle.partitions`?

2. **BroadcastHashJoin memory math:** If `autoBroadcastJoinThreshold=10MB` and you have 200 executors each with 4GB memory, what is the ACTUAL memory consumed by a 10MB broadcast table across the cluster? Why is the in-memory size different from the on-disk size? What configuration limits total broadcast memory?

3. **AQE broadcast conversion:** AQE can convert SortMergeJoin to BroadcastHashJoin at runtime after seeing shuffle statistics. Describe the exact mechanism: what statistic does AQE measure? At what point in execution does the conversion happen? Is any already-completed shuffle work wasted?

4. **Hint priority:** If you apply both a `broadcast` hint and a `merge` hint to the same join, which wins? If CBO strongly recommends BroadcastHashJoin but the developer applied a `merge` hint, which executes? Walk through Spark's complete hint resolution priority order.

5. **Non-equi joins:** A query has `JOIN ON A.start_date <= B.event_date AND A.end_date >= B.event_date`. No equi-join condition. Which strategy does Spark choose? What is the time complexity? What is the memory requirement? How would you rewrite this as an equi-join?

6. **Skew in SortMergeJoin:** In a join on `city`, 95% of rows have `city='NYC'`. One giant task. Describe AQE's `OptimizeSkewedJoin`: how does it detect the skew, how does it split the skewed partition, and what additional data movement does this require from the non-skewed side?

7. **BroadcastNestedLoopJoin vs CartesianProduct:** When is BNLJ chosen vs CartesianProduct? What join condition types trigger BNLJ? Is BNLJ ever acceptable in production, or is it always a sign of a missing equi-join condition?

8. **NULL join key semantics:** In SQL, `NULL = NULL` is false. How does each join strategy (BHJ, SMJ, SHJ) handle NULL keys? Is there a way to join on NULL-equality in Spark? When is NULL-safe join intentional (hint: deduplication via `NULL-safe equality operator <=>`)?

### Expected Logical Plan Discussion

All five join strategies share an identical logical plan — strategy selection is a Physical Planning concern, not a Logical Optimization concern:
```
Project [...]
+- Join Inner, (LicenseType = LicenseType)    ← strategy-agnostic node
   :- Relation cabs_left
   +- Relation cabs_right / tiny_lookup
```

The `Join` logical node is converted to a physical `SparkPlan` by `JoinSelection`, which is a `Strategy` (not a `Rule[LogicalPlan]`). This is a critical interview distinction:
- **Rules** transform `LogicalPlan → LogicalPlan` (same tree type)
- **Strategies** transform `LogicalPlan → Seq[SparkPlan]` (cross-type transformation)

`JoinSelection` generates multiple candidate physical plans and returns the first match. No cost model is used at this point — it is a conditional cascade based on hints, thresholds, and flags.

**For non-equi join:** The logical plan has a non-equality join condition:
```
Join Inner, contains(LicenseType, LicenseType)   ← no equi-join key
```
`JoinSelection` checks: can this be a BHJ? No — needs hash key. Can it be SMJ? No — needs sort key. Falls through to `BroadcastNestedLoopJoinExec`.

### Expected Physical Plan Discussion

**1. BroadcastHashJoin:**
```
*(2) BroadcastHashJoin [LicenseType],[LicenseType], Inner, BuildRight
:- *(2) Filter / scan (probe side — streams, no shuffle)
+- BroadcastExchange HashedRelationBroadcastMode
   +- *(1) LocalTableScan [LicenseType, Code]   (build side — hashed, sent to all executors)
```
- `BuildRight` = right side is hashed into a `HashedRelation` (a Java HashMap)
- Zero shuffle on the probe (left) side — biggest performance advantage
- Memory: `size_of_build_table × num_executors` — watch for cluster-wide OOM on large tables

**2. SortMergeJoin:**
```
*(3) SortMergeJoin [LicenseType],[LicenseType], Inner
:- *(3) Sort [LicenseType ASC NULLS FIRST]
:  +- Exchange hashpartitioning(LicenseType, 200)   ← shuffle left
:     +- *(1) scan left
+- *(3) Sort [LicenseType ASC NULLS FIRST]
   +- Exchange hashpartitioning(LicenseType, 200)   ← shuffle right
      +- *(2) scan right
```
- Two `Exchange` nodes = two full network shuffles = dominant cost for large datasets
- Sort is O(n log n) but uses Tungsten's radix sort for fixed-length types
- Merge phase is O(n+m) streaming — very low memory, handles data larger than RAM

**3. ShuffleHashJoin:**
```
ShuffleHashJoin [LicenseType],[LicenseType], Inner, BuildLeft
:- Exchange hashpartitioning(LicenseType, 200)
:  +- *(1) scan build side
+- Exchange hashpartitioning(LicenseType, 200)
   +- *(2) scan probe side
```
- Same two shuffles as SMJ but NO sort — faster when sort dominates SMJ cost
- Build side partition must fit in executor memory: `build_size / shuffle_partitions < execution_memory_per_executor`
- Not wrapped in `*(N)` — SHJ does not support WholeStageCodegen in all versions

**4. BroadcastNestedLoopJoin:**
```
BroadcastNestedLoopJoin BuildRight, Inner, [condition]
:- *(1) scan left (probe — for every row, scan entire build side)
+- BroadcastExchange IdentityBroadcastMode
   +- *(1) scan right (build — entire table broadcast)
```
- O(n × m) comparisons — catastrophic for large tables
- `IdentityBroadcastMode` (not hashed) because no hash key available

**5. CartesianProduct:**
```
CartesianProduct
:- *(1) LocalTableScan (left)
+- *(1) LocalTableScan (right)
```
- Entire right side replicated to every task — O(n × m) output rows

### Spark UI Investigation Guide — Join Strategy Debugging

**SQL Tab → DAG — count Exchange boxes:**
- 0 Exchange boxes = BroadcastHashJoin (ideal)
- 1 BroadcastExchange = BHJ (small side broadcast)
- 2 ShuffleExchange boxes = SortMergeJoin or ShuffleHashJoin (expensive)

**Stages Tab — per-join-strategy signatures:**

| Metric | BHJ | SMJ | SHJ |
|--------|-----|-----|-----|
| Shuffle Read | 0 on probe side | Large both sides | Large both sides |
| Shuffle Write | 0 | Large | Large |
| Peak Exec Memory | Build hash table size | Low (streaming) | Build partition size |
| GC Time | Low | Low-Medium | High if OOM pressure |
| Number of Tasks | = probe partitions | = shuffle_partitions (×2 stages) | = shuffle_partitions (×2 stages) |

**Diagnosing wrong strategy:**

1. **Expected BHJ, got SMJ:**
   - Check `spark.sql.autoBroadcastJoinThreshold` — is it -1?
   - Check plan statistics: `sizeInBytes=8.0 EiB` means no stats → CBO can't confirm small side
   - Check if AQE is enabled — it may convert at runtime; look for "reused exchange" in UI

2. **Got BHJ but table was large (OOM):**
   - Stats were stale (ANALYZE ran before a large data load)
   - Fix: `spark.sql.autoBroadcastJoinThreshold = -1` + explicit `F.broadcast()` only for verified-small tables

3. **SHJ OOM — one executor dies mid-join:**
   - Build partition exceeded executor execution memory
   - Fix: increase `spark.sql.shuffle.partitions` to reduce per-partition size
   - Or switch to SMJ with `spark.sql.join.preferSortMergeJoin=true`

**War Story:** 500GB fact table joining 15MB dimension ran as SMJ (two 500GB shuffles = 1TB network I/O). Setting `autoBroadcastJoinThreshold=20971520` (20MB) converted to BHJ. Runtime: 45 min → 8 min. Shuffle I/O: 1TB → 0.

In [ ]:
# ── Topic 4: Force Each Join Strategy with Hints ─────────────────────────────
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

# 1. BroadcastHashJoin
print("=" * 60)
print("1. BROADCAST HASH JOIN — F.broadcast() hint")
print("=" * 60)
df_bhj = cabs_left.join(F.broadcast(tiny_lookup), "LicenseType", "inner")
df_bhj.explain()

# 2. SortMergeJoin
print("=" * 60)
print("2. SORT MERGE JOIN — .hint('merge')")
print("=" * 60)
df_smj = cabs_left.hint("merge").join(cabs_right.hint("merge"), "LicenseType")
df_smj.explain()

# 3. ShuffleHashJoin
print("=" * 60)
print("3. SHUFFLE HASH JOIN — .hint('shuffle_hash')")
print("=" * 60)
df_shj = cabs_left.hint("shuffle_hash").join(
    cabs_right.hint("shuffle_hash"), "LicenseType")
df_shj.explain()

# 4. BroadcastNestedLoopJoin (non-equi condition)
print("=" * 60)
print("4. BROADCAST NESTED LOOP JOIN — non-equi condition")
print("=" * 60)
df_bnlj = cabs_left.join(
    tiny_lookup,
    cabs_left.LicenseType.contains(tiny_lookup.LicenseType),
    "inner")
df_bnlj.explain()

# 5. CartesianProduct
print("=" * 60)
print("5. CARTESIAN PRODUCT — explicit crossJoin()")
print("=" * 60)
boroughs_df = spark.createDataFrame([("BK",),("MN",)], ["Borough"])
df_cart = tiny_lookup.crossJoin(boroughs_df)
df_cart.explain()

# Re-enable AQE
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

In [ ]:
# ── Topic 4: Benchmark — BHJ vs SMJ ──────────────────────────────────────────
import time

cabs_left.cache(); cabs_left.count()
tiny_lookup.cache(); tiny_lookup.count()

join_tests = [
    ("BroadcastHashJoin",
     lambda: cabs_left.join(F.broadcast(tiny_lookup), "LicenseType")),
    ("SortMergeJoin (forced)",
     lambda: cabs_left.hint("merge").join(tiny_lookup.hint("merge"), "LicenseType")),
]

print(f"{'Strategy':<28} {'Rows':>10} {'Time':>8} {'vs BHJ':>8}")
print("-" * 58)
baseline = None
for name, build_df in join_tests:
    df = build_df()
    df.count()  # warm-up
    t0 = time.time()
    n = df.count()
    elapsed = time.time() - t0
    if baseline is None:
        baseline = elapsed
        ratio = "1.00x"
    else:
        ratio = f"{elapsed/baseline:.2f}x"
    print(f"  {name:<26} {n:>10,} {elapsed:>7.2f}s {ratio:>8}")

print()
print("BHJ eliminates ALL shuffle I/O on the probe side.")
print("At 500GB scale: BHJ = 0 shuffle; SMJ = 1TB+ shuffle across network.")

### Production Best Practices — Join Strategy Selection

1. **Use explicit `F.broadcast()` hints for all tables you know are small.** Never rely solely on `autoBroadcastJoinThreshold` — it depends on estimated size which can be wrong. `F.broadcast()` is self-documenting and immune to statistics staleness.

2. **Keep `spark.sql.join.preferSortMergeJoin=true` in production.** This prevents accidental ShuffleHashJoin on skewed data that causes OOM. Only override with explicit `shuffle_hash` hints in controlled scenarios with verified even data distribution.

3. **SortMergeJoin is the safe default for large-large joins.** It has O(1) memory usage during the merge phase (streaming). ShuffleHashJoin is faster but requires the entire build partition in memory — one skewed partition causes OOM.

4. **AQE broadcast conversion is a safety net, not a strategy.** If AQE converts SMJ → BHJ at runtime, you already paid the shuffle cost. Plan for broadcast statically whenever possible; treat AQE conversion as a nice bonus for unexpectedly small tables.

5. **Enable AQE skew join handling: `spark.sql.adaptive.skewJoin.enabled=true`.** This is the most impactful single AQE feature for production join workloads. It detects partitions > `spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes` (256MB default) and splits them.

6. **Non-equi joins are O(n×m) — rewrite as equi-joins.** For range joins `A.start <= B.ts AND A.end >= B.ts`, bucket both tables on a discrete time interval (e.g., hour), join on the bucket key, then filter on the exact condition. This converts BNLJ to BHJ.

7. **Monitor `BroadcastExchange` serialization time in the SQL tab.** If broadcasting takes >10s, the table is larger than expected. Check actual size vs estimated size. If actual > `autoBroadcastJoinThreshold`, AQE's broadcast conversion may have triggered unexpectedly.

8. **Use `EXPLAIN FORMATTED` for production plan audits.** The formatted output includes operator IDs, statistics per node, and join strategy labels in tabular form. Easier to parse in CI pipeline logs than tree format.

9. **Bucketing on join keys eliminates shuffle for SMJ.** If two tables are bucketed on the same key with the same number of buckets, SMJ requires zero shuffle (both sides are already co-partitioned). For tables joined repeatedly (e.g., daily ETL), bucketing pays back quickly.

10. **Document join strategy choices in code.** `df.join(F.broadcast(dim), key)  # 5MB dim table — always broadcast`. Prevents future engineers removing the hint, causing a silent performance regression when the job scales up.

### Follow-up Questions — Topic 4

- In Spark 3.3+, `spark.sql.adaptive.optimizeSkewsInRebalancePartitions.enabled` was added. How does this relate to skewed join optimization and when does it trigger vs `OptimizeSkewedJoin`?
- A join between two 500GB tables partitioned by the same key in HDFS. Is it possible to avoid a shuffle in SortMergeJoin? What is a "bucket join" and what are the exact requirements for a bucket join to activate (number of buckets, sort order)?
- What is the `spark.sql.broadcastTimeout` (default 300s) and what happens when it expires? How do you diagnose and fix a broadcast timeout in production?

In [ ]:
# ── Topic 4: Expert Solution — Join Strategy Advisor ─────────────────────────

def join_strategy_advisor(left_mb, right_mb,
                          has_equi_condition=True,
                          executor_memory_gb=4, executor_count=10,
                          shuffle_partitions=200,
                          auto_broadcast_mb=10):
    """
    Recommends a join strategy based on sizes and cluster config.
    Mirrors Spark's JoinSelection logic with production safety margins.
    """
    small_mb = min(left_mb, right_mb)
    large_mb = max(left_mb, right_mb)

    # Execution memory = ~60% of executor memory (Spark unified memory model)
    exec_mem_mb = executor_memory_gb * 1024 * 0.6
    partition_budget_mb = exec_mem_mb / shuffle_partitions  # per-partition budget

    if not has_equi_condition:
        if small_mb < 100:
            rec = "BroadcastNestedLoopJoin"
            notes = [f"No equi-join key. Small side ({small_mb}MB) broadcast.",
                     f"WARNING: O(n x m) comparisons. Rewrite as equi-join if possible."]
        else:
            rec = "DANGER: CartesianProduct or very slow BNLJ"
            notes = ["Non-equi join on large tables will OOM or run indefinitely.",
                     "Redesign query: add equi-join key or bucket on discrete interval."]
    elif small_mb <= auto_broadcast_mb:
        rec = "BroadcastHashJoin (auto or hint)"
        notes = [f"Small side ({small_mb}MB) <= autoBroadcastJoinThreshold ({auto_broadcast_mb}MB)",
                 f"Cluster memory for broadcast: {small_mb * executor_count}MB total",
                 "Code: df.join(F.broadcast(small_df), key)"]
    elif small_mb <= 200:
        rec = "BroadcastHashJoin (explicit hint recommended)"
        notes = [f"Small side ({small_mb}MB) above auto threshold — use explicit F.broadcast()",
                 f"Set autoBroadcastJoinThreshold={small_mb * 1024 * 1024} if preferred",
                 f"Verify executor memory headroom: {small_mb}MB × {executor_count} = {small_mb*executor_count}MB"]
    elif small_mb / shuffle_partitions <= partition_budget_mb * 0.5:
        rec = "ShuffleHashJoin (if sort is dominant cost)"
        notes = [f"Build partition: {small_mb/shuffle_partitions:.1f}MB < {partition_budget_mb*0.5:.0f}MB budget",
                 "Use .hint('shuffle_hash') explicitly",
                 "Keep preferSortMergeJoin=true unless you have verified even distribution"]
    else:
        rec = "SortMergeJoin (default — safe choice)"
        notes = [f"Both sides too large for broadcast/SHJ",
                 f"Total shuffle I/O: {left_mb + right_mb:,}MB",
                 "Enable AQE for skew handling: spark.sql.adaptive.skewJoin.enabled=true",
                 "Consider bucketing both tables on join key to eliminate shuffle"]

    print(f"\n{'='*55}")
    print(f"  Left: {left_mb:>8,}MB   Right: {right_mb:>8,}MB")
    print(f"  Cluster: {executor_count} executors × {executor_memory_gb}GB, "
          f"{shuffle_partitions} shuffle partitions")
    print(f"\n  RECOMMENDATION: {rec}")
    for note in notes:
        print(f"    - {note}")

join_strategy_advisor(1,       0.001)                              # lookup join
join_strategy_advisor(50_000,  15,    executor_memory_gb=16, executor_count=20)
join_strategy_advisor(10_000,  8_000, executor_memory_gb=32,
                      executor_count=100, shuffle_partitions=2000) # large-large
join_strategy_advisor(1_000,   500,   has_equi_condition=False)   # non-equi

## Topic 5 — Cluster Sizing

### Scenario
You are designing a cluster for a new data platform at a logistics company. Requirements: process 10TB of cab trip data daily, SLA = 30 minutes end-to-end, mixed workload of ETL + SQL analytics. You must derive every number from first principles.

### Cluster Sizing Formula Framework
```
STEP 1: Parallelism requirement
  Target tasks      = SLA_seconds / avg_task_seconds
  Min partitions    = target_tasks  (assumes 1 wave)
  Input partitions  = data_bytes / target_partition_size (128MB default)

STEP 2: Core requirement
  Cores needed      = partitions / waves_acceptable
  Waves             = how many rounds of tasks to run
  Recommended: 2-3 waves (allows stragglers without blocking)

STEP 3: Executor sizing (empirical rule: 5 cores per executor)
  Why 5 cores?
    - >5 cores: HDFS throughput degrades (3 replicas contention)
    - <3 cores: executor overhead (JVM startup, Py4J) dominates
    - 5 is the sweet spot validated empirically at Netflix/Databricks

STEP 4: Memory per executor
  executor_memory   = (node_memory - OS_overhead_1GB) / executors_per_node
  Adjust for:
    - storage fraction (caching): spark.memory.fraction = 0.6
    - overhead:  spark.executor.memoryOverhead = max(384MB, 0.1 * executor_memory)
    - off-heap (if enabled): spark.memory.offHeap.size

STEP 5: Parallelism validation
  spark.default.parallelism     = total_cores × 2  (for RDD operations)
  spark.sql.shuffle.partitions  = max(200, total_cores × 3)  (for SQL/DataFrame)

STEP 6: Driver sizing
  driver_memory = max(4GB, largest_broadcast_table × 3)
  Driver holds: broadcast variables, result data from collect(), DAG metadata
```

In [ ]:
# ── Topic 5: Cluster Sizing Calculator ───────────────────────────────────────
# Demonstrate current cluster config and validate against sizing formulas

print("=== Current Spark Session Configuration ===")
configs = [
    ("spark.executor.cores",              "Cores per executor"),
    ("spark.executor.memory",             "Executor JVM heap"),
    ("spark.executor.memoryOverhead",     "Non-heap overhead (Py4J, off-heap buffers)"),
    ("spark.driver.memory",               "Driver heap"),
    ("spark.sql.shuffle.partitions",      "Shuffle partitions (SQL)"),
    ("spark.default.parallelism",         "Default parallelism (RDD)"),
    ("spark.memory.fraction",             "Execution+Storage fraction of heap"),
    ("spark.memory.storageFraction",      "Storage fraction within unified memory"),
    ("spark.sql.autoBroadcastJoinThreshold", "Broadcast threshold"),
]
for key, desc in configs:
    try:
        val = spark.conf.get(key)
    except Exception:
        val = "(not set — using default)"
    print(f"  {key:<45} = {val}")
    print(f"    # {desc}")

print()

# Show active executor info via SparkContext
sc = spark.sparkContext
print(f"  Active executors    : {sc.defaultParallelism} default parallelism")
print(f"  Spark UI            : http://localhost:4040")

### Problem Statement
Given: 10TB of data, 30-minute SLA, each task takes 2 minutes. Derive the complete cluster specification from first principles, showing every formula step.

In [ ]:
# ── Topic 5: Cluster Sizing — First Principles Calculator ────────────────────

def design_cluster(
    data_tb,
    sla_minutes,
    avg_task_minutes=2,
    target_partition_mb=128,
    cores_per_executor=5,
    node_memory_gb=64,
    node_cores=16,
    os_overhead_gb=1,
    waves=2,           # allow 2 waves of tasks (straggler resilience)
    compress_ratio=3,  # Parquet compressed vs in-memory expansion
):
    """
    Full cluster sizing derivation for Spark architects.
    All formulas follow the Netflix/Databricks empirical model.
    """
    print("=" * 65)
    print(f"CLUSTER DESIGN: {data_tb}TB data, {sla_minutes}min SLA")
    print("=" * 65)

    # ── Step 1: Data partitioning ────────────────────────────────
    data_bytes      = data_tb * 1024**4
    partition_bytes = target_partition_mb * 1024**2
    num_partitions  = int(data_bytes / partition_bytes)
    print(f"\nStep 1 — Partitioning:")
    print(f"  Data size              : {data_tb} TB = {data_bytes/1e12:.0f}GB")
    print(f"  Target partition size  : {target_partition_mb} MB")
    print(f"  Input partitions       : {num_partitions:,}")

    # ── Step 2: Parallelism (task throughput) ────────────────────
    sla_seconds        = sla_minutes * 60
    tasks_per_second   = 1 / (avg_task_minutes * 60)
    total_tasks_needed = num_partitions * waves  # assume each partition = 1 task
    cores_needed       = int(num_partitions / waves)  # cores to finish in 1 wave
    print(f"\nStep 2 — Parallelism:")
    print(f"  SLA                    : {sla_minutes} min = {sla_seconds}s")
    print(f"  Avg task time          : {avg_task_minutes} min")
    print(f"  Waves allowed          : {waves}")
    print(f"  Cores needed           : {num_partitions} partitions / {waves} waves = {cores_needed:,} cores")

    # ── Step 3: Executor sizing ──────────────────────────────────
    executors_needed   = int(cores_needed / cores_per_executor)
    executors_per_node = (node_cores - 1) // cores_per_executor  # -1 for YARN NodeManager
    nodes_needed       = int(executors_needed / max(1, executors_per_node)) + 1
    total_executors    = nodes_needed * executors_per_node
    total_cores        = total_executors * cores_per_executor
    print(f"\nStep 3 — Executor sizing:")
    print(f"  Cores per executor     : {cores_per_executor}  (empirical sweet spot)")
    print(f"  Executors needed       : {executors_needed:,}")
    print(f"  Node spec              : {node_cores} cores, {node_memory_gb}GB RAM")
    print(f"  Executors per node     : ({node_cores} - 1 for OS) / {cores_per_executor} = {executors_per_node}")
    print(f"  Nodes needed           : {nodes_needed}")
    print(f"  Total executors        : {total_executors}")
    print(f"  Total cores            : {total_cores:,}")

    # ── Step 4: Memory per executor ─────────────────────────────
    usable_mem_gb      = node_memory_gb - os_overhead_gb
    mem_per_exec_gb    = usable_mem_gb / executors_per_node
    overhead_mb        = max(384, int(mem_per_exec_gb * 1024 * 0.1))
    heap_gb            = mem_per_exec_gb
    exec_mem_gb        = heap_gb * 0.6  # spark.memory.fraction
    storage_mem_gb     = exec_mem_gb * 0.5  # spark.memory.storageFraction
    print(f"\nStep 4 — Memory per executor:")
    print(f"  Usable node memory     : {usable_mem_gb}GB ({node_memory_gb}GB - {os_overhead_gb}GB OS)")
    print(f"  Executor heap          : {mem_per_exec_gb:.1f}GB  (set spark.executor.memory={int(mem_per_exec_gb)}g)")
    print(f"  memoryOverhead         : {overhead_mb}MB  (set spark.executor.memoryOverhead={overhead_mb}m)")
    print(f"  Execution memory       : {exec_mem_gb:.1f}GB  (60% of heap)")
    print(f"  Storage (cache) memory : {storage_mem_gb:.1f}GB  (50% of execution)")
    print(f"  Total per-executor ask : {mem_per_exec_gb:.1f}GB heap + {overhead_mb/1024:.2f}GB overhead")

    # ── Step 5: Shuffle partitions ──────────────────────────────
    shuffle_parts      = max(200, total_cores * 3)
    print(f"\nStep 5 — Parallelism configs:")
    print(f"  spark.sql.shuffle.partitions    = {shuffle_parts}")
    print(f"  spark.default.parallelism       = {total_cores * 2}")

    # ── Step 6: Memory in-use estimate ──────────────────────────
    in_memory_gb = data_tb * 1024 * compress_ratio
    total_storage_mem = total_executors * storage_mem_gb
    cache_fraction = min(1.0, total_storage_mem / in_memory_gb)
    print(f"\nStep 6 — Cache capacity:")
    print(f"  Data in-memory (3x expand) : {in_memory_gb:.0f}GB")
    print(f"  Total cluster storage mem  : {total_storage_mem:.0f}GB")
    print(f"  Cacheable fraction         : {cache_fraction*100:.0f}%")

    # ── Summary ─────────────────────────────────────────────────
    print(f"\n{'='*65}")
    print("FINAL CLUSTER SPECIFICATION:")
    print(f"  Nodes             : {nodes_needed}")
    print(f"  Node spec         : {node_cores} vCPU, {node_memory_gb}GB RAM")
    print(f"  Total executors   : {total_executors}")
    print(f"  Total cores       : {total_cores:,}")
    print(f"  spark.executor.cores   = {cores_per_executor}")
    print(f"  spark.executor.memory  = {int(mem_per_exec_gb)}g")
    print(f"  spark.executor.memoryOverhead = {overhead_mb}m")
    print(f"  spark.sql.shuffle.partitions  = {shuffle_parts}")
    print(f"  spark.default.parallelism     = {total_cores * 2}")

# The 10TB / 30 min interview scenario
design_cluster(data_tb=10, sla_minutes=30, avg_task_minutes=2,
               node_memory_gb=64, node_cores=16, waves=2)

### Interview Questions — Topic 5: Cluster Sizing

1. **The 5-core rule:** Why is 5 cores per executor the empirical sweet spot? What happens to HDFS throughput with >5 cores per executor? What happens to JVM overhead with <3 cores per executor? Under what workload would you deviate from 5 cores?

2. **Memory math:** A node has 128GB RAM, 32 cores. You want 5 cores per executor. Walk through: how many executors per node? What is executor heap? What is `memoryOverhead`? What is the final YARN memory request per executor (heap + overhead)?

3. **Shuffle partitions formula:** `spark.sql.shuffle.partitions=200` is the default. For a 10TB dataset on a 500-core cluster, is 200 correct? What is the formula for the right number? What are the consequences of too few vs too many shuffle partitions?

4. **AQE `coalesceShufflePartitions`:** AQE can reduce shuffle partitions at runtime by coalescing small post-shuffle partitions. What is `spark.sql.adaptive.coalescePartitions.minPartitionNum`? How does AQE decide which partitions to coalesce? Can coalescing ever make things worse?

5. **Dynamic allocation:** `spark.dynamicAllocation.enabled=true` scales executors up/down. What is the protocol for an executor to be considered idle and returned to the cluster? How does dynamic allocation interact with cached data (if an executor with cached partitions is killed)?

6. **YARN vs Kubernetes resource requests:** On YARN, `spark.executor.memory` + `spark.executor.memoryOverhead` = the total container memory request. On Kubernetes, you must also set `spark.kubernetes.executor.request.cores`. Explain the difference between `request` and `limit` in Kubernetes and why setting them equal is important for Spark.

7. **Partition size sweet spot:** 128MB is the default target partition size. What happens with 10MB partitions (too small)? What happens with 1GB partitions (too large)? What specific Spark operations are most sensitive to partition size?

8. **Cost modeling:** Your team can choose between: (a) 100 nodes × 32-core, 64GB each, or (b) 50 nodes × 64-core, 128GB each. Same total cores and memory. Which do you choose for a Spark workload and why? What does the shuffle network topology have to do with it?

### Expected Logical Plan Discussion — Topic 5

Cluster sizing does not affect logical plans — it affects physical execution throughput and resource allocation. The relevant "plan" to discuss here is the **resource request plan** made to YARN/Kubernetes.

**YARN Resource Request Chain:**
```
spark-submit
  --executor-memory 20g
  --executor-cores 5
  --num-executors 40
  
  YARN sees: 40 container requests, each 20g + 2g overhead = 22g + 5 vCPU
  Total request: 40 × 22g = 880GB RAM, 40 × 5 = 200 vCPU
  YARN schedules on nodes that have 22g + 5 vCPU available
```

**Spark Unified Memory Model** (what happens inside the 20g heap):
```
20g executor heap
├── Reserved (300MB)     — JVM overhead, class metadata
├── Execution Memory     — shuffle buffers, sort buffers, hash tables
│   └── 60% × (20g - 300MB) × (1 - storageFraction)
├── Storage Memory       — cached RDD/DataFrame partitions
│   └── 60% × (20g - 300MB) × storageFraction
└── User Memory          — user objects, UDF state, Python workers
    └── 40% × (20g - 300MB)
```
Execution and Storage memory share a unified pool — either can borrow from the other as needed, subject to eviction of cached data.

### Spark UI Investigation Guide — Cluster Sizing Validation

**Executors Tab** (`/executors`):
- **Active Executors:** Should match `num-executors` (or dynamic allocation target)
- **Task Time:** Total CPU-hours consumed — divide by wall time to get CPU utilization %
- **GC Time:** Should be <10% of Task Time cluster-wide. If GC Time / Task Time > 20%, executors are undersized or data is too large for memory
- **Input / Shuffle Read/Write:** Aggregate these across all executors to get total I/O — use to validate partition size calculations

**Jobs Tab** — Stage durations:
- Total wall time of all stages / (total cores × SLA) = CPU utilization
- If <60%, you are over-provisioned. If >85%, you are under-provisioned or have stragglers.

**Stages Tab** — Task distribution:
- Click a stage → Task Duration distribution histogram
- Ideal: tight bell curve (all tasks similar duration)
- Problem sign: long tail (straggler) — one task 10× longer than median → data skew
- Problem sign: bimodal — two clusters of task times → some tasks doing much more work

**Key ratios to monitor:**
| Metric | Formula | Healthy | Action if bad |
|--------|---------|---------|---------------|
| CPU utilization | Task Time / (executors × cores × stage_time) | 70-90% | Scale down if <60%, up if >90% |
| GC overhead | GC Time / Task Time | <10% | Increase executor memory or enable G1GC |
| Shuffle spill | Shuffle Spill (disk) > 0 | 0 | Increase spark.sql.shuffle.partitions |
| Executor OOM | Executor Lost messages in logs | 0 | Increase executor memory or reduce partition size |

In [ ]:
# ── Topic 5: Parallelism Validation — measure partition sizes on Cabs data ───
import os

# Check how Spark partitions our cabs data
df_check = cabs_raw
print(f"cabs_raw partitions     : {df_check.rdd.getNumPartitions()}")
print(f"cabs_raw row count      : {df_check.count():,}")
print()

# Show the effect of repartition on task count and data distribution
for n_parts in [1, 4, 8, 16]:
    repartitioned = cabs_raw.repartition(n_parts)
    # Collect partition sizes
    sizes = repartitioned.rdd.mapPartitions(
        lambda it: [sum(1 for _ in it)]
    ).collect()
    avg = sum(sizes) / len(sizes)
    max_s = max(sizes)
    min_s = min(sizes)
    print(f"  repartition({n_parts:>2}) → partitions={len(sizes)}, "
          f"avg={avg:.0f}, min={min_s}, max={max_s} rows")

print()
# Show current parallelism settings
print(f"spark.default.parallelism      = "
      f"{spark.conf.get('spark.default.parallelism', str(sc.defaultParallelism))}")
print(f"spark.sql.shuffle.partitions   = "
      f"{spark.conf.get('spark.sql.shuffle.partitions')}")
print()
print("Rule of thumb:")
print("  spark.sql.shuffle.partitions = max(200, total_cores * 3)")
print("  Ideal partition size after shuffle = 100-200MB")
print("  Too small (<10MB): task scheduling overhead dominates")
print("  Too large (>1GB): risk of OOM on sort/join operations")

### Production Best Practices — Cluster Sizing

1. **Start with the 5-core-per-executor rule.** It is empirically validated across hundreds of production clusters at Netflix, Databricks, and Uber. Deviate only when: (a) your workload is pure compute with no HDFS reads (can use more cores), or (b) tasks are memory-intensive (use fewer cores to give each more memory).

2. **Always reserve 1 core per node for OS and YARN NodeManager.** Setting `executors_per_node = (node_cores - 1) / executor_cores` prevents the OS from competing with Spark tasks for CPU, which causes unpredictable task time variance.

3. **`spark.executor.memoryOverhead` is not optional.** Default is `max(384MB, 0.1 × executor_memory)`. For Python workloads (PySpark), the Python worker process runs OUTSIDE the JVM heap — it uses the overhead allocation. Undersizing this causes container kills from YARN/Kubernetes with no JVM OOM exception.

4. **`spark.sql.shuffle.partitions` is the single most impactful tuning knob.** Default 200 is only correct for ~1-2GB datasets on a small cluster. Formula: `max(200, total_cores × 3)`. Use AQE's `coalescePartitions` to handle cases where the actual post-shuffle data is smaller than expected.

5. **Size the driver for the largest `collect()` or broadcast table.** `driver_memory = max(4g, largest_broadcast × 3)`. The driver holds: all broadcast variables (replicated from all executors), all results from `collect()`, the full DAG in memory. A driver OOM kills the entire job.

6. **Use `spark.dynamicAllocation.enabled=true` for mixed workloads.** Static allocation wastes resources when Spark is idle between stages. Dynamic allocation returns executors to the cluster after `spark.dynamicAllocation.executorIdleTimeout` (default 60s). Set `spark.shuffle.service.enabled=true` so idle executors can be removed without losing shuffle data.

7. **Benchmark task time distribution before sizing.** `avg_task_minutes` is a critical input. Profile a representative sample job to get the P50, P90, P99 task duration. Size for P90 (not P50) to handle typical stragglers without SLA breach.

8. **Add 20-30% buffer to all resource calculations.** The formula gives the theoretical minimum. Add buffer for: GC pauses, data skew, task speculation, stage retry, and broadcast variable distribution time.

9. **On Kubernetes: set resource request = resource limit.** If limits > requests, a memory spike causes the pod to be OOM-killed by the kernel (not the JVM) — no `OutOfMemoryError` in logs, just a sudden executor disappearance. Equal request/limit ensures predictable behavior.

10. **Validate your sizing with a pilot job before full production rollout.** Run 10% of data on the proposed cluster. Measure: CPU utilization, GC overhead, shuffle spill, task duration P99. Adjust sizing based on observed metrics. Never extrapolate from theory alone.

### Follow-up Questions — Topic 5

- You have `spark.default.parallelism=1000` and `spark.sql.shuffle.partitions=200`. A Spark SQL query with a GROUP BY runs. Which setting applies? When does each setting apply and why does this commonly cause confusion?
- AQE's `coalesceShufflePartitions` reduced your shuffle partitions from 2000 to 47 at runtime. What does this tell you about your initial sizing of `spark.sql.shuffle.partitions`? Is this always good? Name a scenario where AQE's coalescing makes performance worse.
- Explain the `spark.executor.instances` vs `spark.dynamicAllocation.maxExecutors` interplay. If both are set, which wins? How do you set a hard cap on cluster usage in a shared environment?

In [ ]:
# ── Topic 5: Expert Solution — Cluster Config Validator ──────────────────────

def validate_cluster_config(spark):
    """
    Validates current Spark cluster config against production best practices.
    Outputs a scored report with specific remediation steps.
    """
    issues = []
    score  = 100

    def get_conf(key, default=None):
        try:
            return spark.conf.get(key)
        except Exception:
            return default

    # Executor memory
    exec_mem = get_conf("spark.executor.memory", "1g")
    exec_mem_mb = int(exec_mem.replace("g","000").replace("m",""))
    if "g" in exec_mem:
        exec_mem_mb = int(exec_mem.replace("g","")) * 1024
    overhead = get_conf("spark.executor.memoryOverhead")
    if overhead is None:
        expected_overhead = max(384, int(exec_mem_mb * 0.1))
        issues.append(f"memoryOverhead not set — will default to {expected_overhead}MB; "
                      f"set explicitly to avoid container kill surprises")
        score -= 5

    # Shuffle partitions
    shuffle_parts = int(get_conf("spark.sql.shuffle.partitions", "200"))
    default_par   = int(get_conf("spark.default.parallelism",
                                  str(spark.sparkContext.defaultParallelism)))
    if shuffle_parts == 200:
        issues.append("spark.sql.shuffle.partitions=200 (default) — "
                      "tune to max(200, total_cores * 3) for production workloads")
        score -= 10

    # AQE
    aqe = get_conf("spark.sql.adaptive.enabled", "false")
    if aqe.lower() != "true":
        issues.append("AQE disabled — enable spark.sql.adaptive.enabled=true for "
                      "automatic partition coalescing and skew join handling")
        score -= 15

    # CBO
    cbo = get_conf("spark.sql.cbo.enabled", "false")
    if cbo.lower() != "true":
        issues.append("CBO disabled — enable spark.sql.cbo.enabled=true "
                      "for join reordering based on statistics")
        score -= 10

    # Serializer
    ser = get_conf("spark.serializer", "org.apache.spark.serializer.JavaSerializer")
    if "Kryo" not in ser:
        issues.append("Using JavaSerializer — switch to KryoSerializer for "
                      "3-5x faster RDD serialization: spark.serializer="
                      "org.apache.spark.serializer.KryoSerializer")
        score -= 10

    score = max(0, score)
    grade = "A" if score >= 90 else "B" if score >= 75 else "C" if score >= 60 else "F"

    print(f"\n{'='*60}")
    print(f"Cluster Config Validator")
    print(f"{'='*60}")
    print(f"  Score: {score}/100  Grade: {grade}")
    print(f"  shuffle.partitions    = {shuffle_parts}")
    print(f"  default.parallelism   = {default_par}")
    print(f"  executor.memory       = {exec_mem}")
    print(f"  AQE enabled           = {aqe}")
    print(f"  CBO enabled           = {cbo}")
    print(f"  Serializer            = {ser.split('.')[-1]}")
    if issues:
        print(f"\n  Issues ({len(issues)}):")
        for i, issue in enumerate(issues, 1):
            print(f"    {i}. {issue}")
    else:
        print("\n  Config looks good for production!")

validate_cluster_config(spark)

## Topic 6 — Executor Tuning

### Scenario
A nightly batch job at a payments company runs on a 20-node cluster. The job runs for 3 hours but the team says it "used to run in 45 minutes". The Spark UI shows GC Time is 55% of Task Time across all executors. The executor config is 1 core × 40 executors (fat-thin anti-pattern reversed). You must redesign the executor configuration and GC tuning.

### Fat vs Thin Executors

```
Scenario: 20-node cluster, 64-core, 256GB RAM per node. 
Total resources: 1,280 cores, 5,120GB RAM.

OPTION A: Thin executors (anti-pattern)
  1 core × 1,280 executors = 1,280 executors
  Memory per executor: 5,120GB / 1,280 = 4GB
  Problems:
    - 1,280 JVM processes = massive OS overhead
    - Each executor starts its own Python worker process (1,280 Python processes)
    - Broadcast table replicated 1,280 times
    - HDFS connections: 1,280 × 3 replicas = 3,840 simultaneous connections

OPTION B: Fat executors (anti-pattern)
  64 cores × 20 executors = 1,280 cores (1 executor per node)
  Memory per executor: 256GB - 1GB OS = 255GB
  Problems:
    - >5 cores: HDFS client contention on 3 replicas
    - 255GB heap: Full GC pauses can last minutes
    - One OOM kills the entire node's work — no isolation

OPTION C: Recommended (5 cores, balanced)
  5 cores × (63/5=12) executors × 20 nodes = 240 executors
  Memory per executor: (255GB) / 12 = ~21GB
  Benefits:
    - 240 JVM processes (manageable)
    - HDFS: 5 × 3 = 15 simultaneous reads (within HDFS limits)
    - Broadcast: 240 copies of 21GB each (manageable)
    - 21GB heap: G1GC can handle this without multi-second pauses
```

### JVM GC for Spark: G1GC vs ParallelGC
```
ParallelGC (pre-Spark 3.1 default):
  - Stop-The-World collections: entire executor pauses
  - Full GC (Old Gen): can pause 10-60 seconds for large heaps
  - Triggers: Old Gen > 70% full
  - Bad for Spark: tasks accumulate short-lived objects (shuffle buffers,
    UnsafeRow wrappers) → frequent Young GC → task time variance

G1GC (Spark 3.1+ default, recommended):
  - Concurrent marking: GC runs alongside application threads
  - Region-based heap: Old Gen collected incrementally in regions
  - Predictable pause target: -XX:MaxGCPauseMillis=200
  - Better for Spark: large heap divided into ~2048 regions of 1-32MB each
    regions with mostly-dead objects collected first (Garbage First)
  - set: -XX:+UseG1GC -XX:G1HeapRegionSize=32m -XX:MaxGCPauseMillis=200
```

In [ ]:
# ── Topic 6: Fat vs Thin Executor Comparison ─────────────────────────────────

def compare_executor_configs(node_count, node_cores, node_memory_gb,
                              broadcast_table_mb=100):
    """
    Compare thin, fat, and recommended executor configurations.
    Quantifies resource waste and identifies the GC risk profile.
    """
    os_overhead_gb = 1
    usable_memory  = node_memory_gb - os_overhead_gb

    configs = {
        "Thin (1 core)": {
            "cores_per_exec": 1,
            "executors_per_node": node_cores - 1,
        },
        "Fat (all cores)": {
            "cores_per_exec": node_cores,
            "executors_per_node": 1,
        },
        "Recommended (5 cores)": {
            "cores_per_exec": 5,
            "executors_per_node": (node_cores - 1) // 5,
        },
    }

    print(f"Node spec: {node_count} × {node_cores}-core, {node_memory_gb}GB")
    print(f"{'Config':<25} {'Execs':>6} {'Cores':>6} {'Mem/Exec':>10} "
          f"{'BroadcastMem':>14} {'GC Risk':>10}")
    print("-" * 75)

    for name, cfg in configs.items():
        ep  = cfg["executors_per_node"]
        cpe = cfg["cores_per_exec"]
        total_execs = node_count * ep
        total_cores = total_execs * cpe
        mem_per_exec_gb = usable_memory / ep
        broadcast_cluster_mb = broadcast_table_mb * total_execs

        # GC risk heuristic
        if mem_per_exec_gb < 4:
            gc_risk = "HIGH (tiny heap)"
        elif mem_per_exec_gb > 60:
            gc_risk = "HIGH (huge heap)"
        elif cpe == 1:
            gc_risk = "MED (many JVMs)"
        else:
            gc_risk = "LOW"

        print(f"  {name:<23} {total_execs:>6} {total_cores:>6} "
              f"{mem_per_exec_gb:>8.1f}GB "
              f"{broadcast_cluster_mb/1024:>12.1f}GB  "
              f"{gc_risk:>12}")

    print()
    print("Key insight: Recommended config balances:")
    print("  - Executor count (not too many JVMs)")
    print("  - HDFS throughput (5 threads × 3 replicas = manageable)")
    print("  - Heap size (G1GC works well on 16-32GB heaps)")
    print("  - Broadcast overhead (broadcast × executors = cluster memory)")

compare_executor_configs(
    node_count=20, node_cores=64, node_memory_gb=256,
    broadcast_table_mb=100
)

### Problem Statement
Demonstrate the GC pressure caused by object-heavy operations. Show how to measure GC overhead via Spark metrics and derive the correct G1GC tuning parameters for a production executor configuration.

In [ ]:
# ── Topic 6: GC Pressure Demo — object-heavy vs Tungsten operations ──────────
import time

# GC-heavy: Python UDFs create many short-lived Java objects per row
from pyspark.sql.functions import udf

@udf(returnType=StringType())
def gc_heavy_udf(name, addr, phone):
    """Creates many short-lived Python + Java objects per row.
    Each call: 3 Python string objects + JVM serialization objects
    At 44,850 rows: ~500K+ short-lived objects → Young GC pressure"""
    if name and addr and phone:
        return f"{name.upper()}-{addr[:10]}-{phone.replace('-','')}"
    return "UNKNOWN"

cabs_gc = cabs_t2  # ~44,850 rows

# GC-heavy path
start = time.time()
n_gc = (cabs_gc
    .withColumn("Key", gc_heavy_udf(
        F.col("Name"), F.col("Address"), F.col("TelephoneNumber")))
    .filter(F.col("Key") != "UNKNOWN")
    .count())
t_gc = time.time() - start

# Tungsten-native path (no GC pressure)
start = time.time()
n_native = (cabs_gc
    .withColumn("Key",
        F.concat(
            F.upper(F.col("Name")),
            F.lit("-"),
            F.col("Address").substr(1, 10),
            F.lit("-"),
            F.regexp_replace(F.col("TelephoneNumber"), "-", "")
        ))
    .filter(F.col("Key") != "UNKNOWN")
    .count())
t_native = time.time() - start

print(f"GC-heavy UDF path    : {n_gc:,} rows in {t_gc:.2f}s")
print(f"Tungsten native path : {n_native:,} rows in {t_native:.2f}s")
print(f"Ratio                : {t_gc/max(t_native,0.001):.1f}x slower with UDF")
print()
print("In the Spark UI Stages tab, compare GC Time for each stage.")
print("The UDF stage will show significantly higher GC overhead.")

### Interview Questions — Topic 6: Executor Tuning

1. **GC pressure root cause:** What are the five primary sources of GC pressure in a Spark executor? For each source, explain which GC generation (Young/Old) it primarily affects and what specific Spark operations trigger it.

2. **G1GC region sizing:** You set `-XX:G1HeapRegionSize=32m` on a 21GB executor heap. How many regions does G1GC create? What happens when a single object is larger than one region (humongous object)? How does this relate to Spark's `UnsafeRow` allocation behavior?

3. **Off-heap memory model:** `spark.memory.offHeap.enabled=true` moves execution memory outside the JVM heap. Describe exactly which Spark data structures move off-heap. What CANNOT be moved off-heap? Why does off-heap memory reduce GC pressure even though Spark still runs on the JVM?

4. **`memoryOverhead` for Python:** A PySpark job uses Python UDFs. The executor has `spark.executor.memory=8g` and default `memoryOverhead`. A Python worker is launched per executor core. What is the total non-JVM memory per executor? What happens when `memoryOverhead` is undersized on Kubernetes?

5. **Executor lost vs OOM:** A Spark job shows "ExecutorLostFailure" in the driver log but no `OutOfMemoryError` in the executor log. Name three possible causes and how you distinguish between them in the Spark UI and cluster manager (YARN/Kubernetes) logs.

6. **Task speculation:** `spark.speculation=true` launches speculative copies of slow tasks. How does speculation interact with executor memory? Can speculation CAUSE OOM? Describe the exact mechanism by which aggressive speculation can exhaust executor memory.

7. **Heap vs off-heap caching:** `df.cache()` stores data in JVM on-heap storage by default. What is `MEMORY_AND_DISK_DESER`? What is `OFF_HEAP` persistence level? Compare the GC impact of on-heap vs off-heap caching for a 50GB DataFrame on a cluster with 200 executors × 4GB memory.

8. **GC log analysis:** You receive a GC log file from a production executor that has 55% GC overhead. Describe your analysis process: what log lines do you look for? What is the difference between a Young GC event and a Full GC event in the G1GC log format? What threshold separates "normal" from "catastrophic" GC?

### Expected Logical/Physical Plan Discussion — Topic 6

Executor tuning does not change the query plan — it changes how the plan EXECUTES. The relevant discussion is the **memory allocation model** inside the executor during physical plan execution.

**Memory Allocation During SortMergeJoin (Physical Plan in memory):**
```
SortMergeJoin execution — memory allocation trace:
  1. Exchange (shuffle write):
     → ShuffleMapTask allocates ExternalSorter (uses execution memory)
     → Serializes rows to shuffle files (UnsafeRow → byte array → disk)
     → Short-lived objects: 1 byte[] per shuffle buffer flush
     → GC impact: Young GC on each buffer flush (~32MB default)

  2. Exchange (shuffle read):
     → ShuffleReader fetches blocks from shuffle service
     → Deserializes blocks into memory (UnsafeRow format)
     → Memory pool: execution memory (up to spark.memory.fraction × heap)

  3. Sort:
     → UnsafeExternalSorter allocates sort buffer in execution memory
     → If execution memory insufficient: SPILL to disk
     → GC impact: minimal (UnsafeRow is off-heap or pooled)

  4. Merge:
     → Streaming scan of two sorted iterators
     → Near-zero memory allocation (one row at a time)
     → GC impact: near-zero
```

**Memory during Python UDF (the GC killer):**
```
ArrowEvalPython node execution:
  For each Arrow batch (default 10,000 rows):
    1. JVM: UnsafeRow[] → serialize to Arrow IPC format (byte[])
    2. PIPE: write to Python worker stdin
    3. Python: deserialize Arrow → pandas DataFrame
    4. Python: run UDF on entire batch (vectorized or row-by-row)
    5. Python: serialize result → Arrow IPC
    6. PIPE: write to JVM stdout
    7. JVM: deserialize Arrow → Java object → UnsafeRow

At step 1 and 7: thousands of short-lived Java objects created per batch
→ Young GC triggered every 2-5 batches
→ If UDF result objects survive 2 Young GC cycles → promoted to Old Gen
→ Old Gen fills up → Full GC → 10-60 second pause → 55% GC overhead
```

### Spark UI Investigation Guide — GC Debugging

**Executors Tab** (`/executors`) — the primary GC dashboard:
- **GC Time column:** Cumulative GC time per executor. Sort by this column descending.
  - Healthy: GC Time < 10% of Task Time
  - Concerning: GC Time 10-20% of Task Time
  - Critical: GC Time > 20% (production incidents at >30%)
- **Storage Memory column:** How much of the storage memory pool is used. If >90%, storage is evicting cached partitions to give room to execution → re-reading data from disk → slower jobs.

**Stages Tab** — per-stage GC breakdown:
- Click any stage → **"Aggregated Metrics by Executor"** section
- Look for columns: `GC Time`, `Peak Execution Memory`, `Shuffle Spill (Memory)`, `Shuffle Spill (Disk)`
- **Shuffle Spill (Disk) > 0:** Execution memory insufficient — increase executor memory or shuffle partitions
- **Peak Execution Memory near executor memory limit:** OOM risk in the next run

**Stage Details → Task List:**
- Sort by "GC Time" column — identify individual tasks with abnormally high GC
- If only some tasks have high GC (not all): data skew causing some partitions to be much larger, exhausting their executor's memory
- If ALL tasks have high GC: systematic under-provisioning of executor memory or GC algorithm mismatch

**Recommended GC JVM flags to add to `spark.executor.extraJavaOptions`:**
```
-XX:+UseG1GC
-XX:G1HeapRegionSize=32m          # for 21GB+ heaps
-XX:MaxGCPauseMillis=200          # soft pause target (not guaranteed)
-XX:InitiatingHeapOccupancyPercent=35  # start concurrent mark earlier
-XX:+PrintGCDetails                # for log analysis
-XX:+PrintGCDateStamps
-Xss4m                             # larger thread stack for deep recursion
```

**War Story:** Payments company's nightly job regressed from 45 min to 3 hours. Root cause: a developer added a Python UDF to normalize phone numbers. This single UDF caused 55% GC overhead. Fix: replaced the UDF with `F.regexp_replace(col, "[^0-9]", "")`. Job returned to 48 minutes. Lesson: a single UDF in a pipeline processing 500M rows daily = millions in annual compute cost.

In [ ]:
# ── Topic 6: Off-Heap Memory Demo ────────────────────────────────────────────
# Show how off-heap configuration changes the memory model

print("=== Off-Heap Memory Configuration ===\n")
print("CURRENT settings:")
for key in ["spark.memory.offHeap.enabled",
            "spark.memory.offHeap.size",
            "spark.memory.fraction",
            "spark.memory.storageFraction"]:
    try:
        v = spark.conf.get(key)
    except Exception:
        v = "(not set)"
    print(f"  {key} = {v}")

print()
print("RECOMMENDED for shuffle-heavy workloads (>500GB daily shuffle):")
print("  spark.memory.offHeap.enabled = true")
print("  spark.memory.offHeap.size    = <50% of executor memory>")
print("  Example for 20g executor:")
print("    spark.executor.memory    = 20g")
print("    spark.memory.offHeap.size = 10737418240  # 10GB in bytes")
print()
print("WHY off-heap reduces GC:")
print("  - Shuffle sort buffers → stored in off-heap pages (UnsafeExternalSorter)")
print("  - Hash tables for aggregation → stored in off-heap (BytesToBytesMap)")
print("  - JVM heap only holds: task objects, UDF closures, Python worker comms")
print("  - GC scans ONLY the reduced JVM heap → faster, shorter pauses")
print()
print("RISK: off-heap memory is outside JVM GC. If executor crashes,")
print("  off-heap memory may not be freed. Use with shuffle service enabled.")

# Demonstrate with caching persistence levels
from pyspark import StorageLevel

print()
print("=== Storage Persistence Levels ===")
for level_name, level in [
    ("MEMORY_ONLY (default for .cache())",        StorageLevel.MEMORY_ONLY),
    ("MEMORY_AND_DISK",                           StorageLevel.MEMORY_AND_DISK),
    ("MEMORY_ONLY_SER (Java serialized)",         StorageLevel.MEMORY_ONLY_SER),
    ("OFF_HEAP (requires offHeap enabled)",       StorageLevel.OFF_HEAP),
]:
    print(f"  {level_name}")
    print(f"    useDisk={level.useDisk}, useMemory={level.useMemory}, "
          f"useOffHeap={level.useOffHeap}, deserialized={level.deserialized}")

### Production Best Practices — Executor Tuning

1. **5 cores per executor is the starting point, not the rule.** Benchmark your specific workload. I/O-bound jobs (reading from S3 with high latency) benefit from more cores per executor (7-10) to keep more concurrent I/O requests in flight. Compute-bound jobs with large in-memory sorts should use fewer cores (3-4) to give each task more heap headroom.

2. **Use G1GC for executors with >8GB heap.** `spark.executor.extraJavaOptions=-XX:+UseG1GC -XX:G1HeapRegionSize=32m -XX:MaxGCPauseMillis=200`. G1GC's concurrent collection eliminates the multi-second Full GC pauses that ParallelGC produces on large heaps. For heaps <8GB, ParallelGC may actually be faster due to lower G1GC overhead.

3. **`-XX:InitiatingHeapOccupancyPercent=35` (default 45) for Spark.** Lowering IHOP makes G1GC start concurrent marking earlier, before the heap fills up. Spark tasks fill heap quickly with shuffle buffers, so earlier concurrent marking prevents fallback Full GCs. Monitor: if you see "to-space exhausted" in GC logs, lower IHOP further.

4. **Enable off-heap for shuffle-heavy workloads.** `spark.memory.offHeap.enabled=true` with `spark.memory.offHeap.size` set to 50% of executor memory. The JVM heap only needs to hold task metadata and closures — all bulk data lives off-heap. GC pauses drop dramatically because the heap is smaller and mostly contains long-lived objects.

5. **`spark.executor.memoryOverhead` must account for Python workers.** Each executor core can spawn one Python worker process. A 5-core executor with 200MB Python processes = 1GB of non-JVM memory. Set `memoryOverhead = max(512MB, 0.15 × executor_memory)` for PySpark jobs.

6. **Never run production jobs with `MEMORY_ONLY` caching on large DataFrames.** If the DataFrame doesn't fit in memory, Spark silently drops cached partitions and re-reads from disk with no error. Use `MEMORY_AND_DISK` in production to guarantee data is always available, even if evicted.

7. **Tune Young Gen size for Spark's object lifetime pattern.** Spark tasks create many short-lived objects (shuffle buffers, intermediate Row objects) that should die in Young GC. Add `-XX:NewRatio=1` to make Young Gen 50% of heap (default is 33%). More Young Gen = fewer promotions to Old Gen = less Full GC.

8. **Enable GC logging in production for 30-day baseline.** `-XX:+PrintGCDetails -XX:+PrintGCDateStamps -Xloggc:/tmp/executor-gc.log`. Parse with GCViewer or Gceasy.io. Establish baseline GC overhead %. Alert when a deploy causes >5% increase — this indicates an object-heavy code change.

9. **Increase `spark.shuffle.file.buffer` to reduce flush frequency.** Default is 32KB — very small. For large shuffle stages, set to 1MB: `spark.shuffle.file.buffer=1m`. Larger buffer = fewer write syscalls = fewer short-lived byte arrays = less Young GC during shuffle write.

10. **Off-heap caching (`StorageLevel.OFF_HEAP`) for repeatedly-accessed DataFrames.** If a 50GB DataFrame is cached on-heap across 200 executors, each executor holds ~250MB in JVM heap as `CachedBatch` objects. This is a GC burden even when not being accessed. Off-heap caching removes it from the GC scan entirely.

### Follow-up Questions — Topic 6

- ZGC (Z Garbage Collector, JDK 11+) promises sub-millisecond pause times. Is ZGC better than G1GC for Spark? What are ZGC's tradeoffs (throughput vs latency) and when would you choose ZGC over G1GC for a Spark cluster?
- `spark.executor.extraJavaOptions` applies to all tasks in an executor. Can you have different GC settings per stage? Per job? What is the scope of these JVM flags?
- An executor's GC logs show "G1 Evacuation Pause (mixed)" events taking 2 seconds each, occurring every 30 seconds. What does "mixed" mean in G1GC terminology? What does it indicate about your heap's Old Gen?

In [ ]:
# ── Topic 6: Expert Solution — GC Tuning Config Generator ───────────────────

def generate_gc_config(executor_memory_gb, cores_per_executor,
                       has_python_udfs=True, shuffle_heavy=False):
    """
    Generates production-grade GC tuning configuration for Spark executors.
    All settings are derived from the executor memory size and workload type.
    """
    heap_gb      = executor_memory_gb
    young_gb     = heap_gb // 2      # NewRatio=1 → 50% Young Gen
    region_mb    = 32 if heap_gb >= 16 else 16  # G1 region size

    overhead_mb  = max(512, int(heap_gb * 1024 * 0.15)) if has_python_udfs \
                   else max(384, int(heap_gb * 1024 * 0.1))

    # GC algorithm selection
    if heap_gb >= 8:
        gc_flags = [
            "-XX:+UseG1GC",
            f"-XX:G1HeapRegionSize={region_mb}m",
            "-XX:MaxGCPauseMillis=200",
            "-XX:InitiatingHeapOccupancyPercent=35",
            "-XX:+G1UseAdaptiveIHOP",
            "-XX:NewRatio=1",  # 50% Young Gen
        ]
        gc_note = "G1GC (recommended for large heaps)"
    else:
        gc_flags = [
            "-XX:+UseParallelGC",
            "-XX:NewRatio=2",
        ]
        gc_note = "ParallelGC (faster for small heaps <8GB)"

    # Universal flags
    universal_flags = [
        "-XX:+PrintGCDetails",
        "-XX:+PrintGCDateStamps",
        "-Xss4m",                         # thread stack size
        "-XX:+UseStringDeduplication",    # reduce String object overhead (G1 only)
    ]
    if shuffle_heavy:
        universal_flags.append("-XX:+AlwaysPreTouch")  # pre-allocate heap pages

    all_flags = gc_flags + universal_flags
    flags_str = " ".join(all_flags)

    print(f"\n{'='*60}")
    print(f"GC Config: {heap_gb}GB heap, {cores_per_executor} cores, "
          f"Python={'YES' if has_python_udfs else 'NO'}")
    print(f"{'='*60}")
    print(f"  GC Algorithm      : {gc_note}")
    print(f"  G1 region size    : {region_mb}MB  ({heap_gb*1024//region_mb} regions)")
    print(f"  Young Gen         : {young_gb}GB of {heap_gb}GB heap")
    print(f"  memoryOverhead    : {overhead_mb}MB")
    print()
    print("  spark-submit flags:")
    print(f"    --executor-memory {heap_gb}g")
    print(f"    --conf spark.executor.memoryOverhead={overhead_mb}m")
    print(f"    --conf spark.executor.extraJavaOptions='{flags_str}'")
    print()
    print("  spark-defaults.conf:")
    print(f"    spark.executor.memory                = {heap_gb}g")
    print(f"    spark.executor.memoryOverhead        = {overhead_mb}m")
    print(f"    spark.executor.extraJavaOptions      = {flags_str}")

generate_gc_config(executor_memory_gb=21, cores_per_executor=5,
                   has_python_udfs=True, shuffle_heavy=True)
generate_gc_config(executor_memory_gb=6, cores_per_executor=5,
                   has_python_udfs=False, shuffle_heavy=False)

## Topic 7 — End-to-End Pipeline Optimization

### Scenario
A logistics company's ETL pipeline reads 5TB of cab data daily, enriches it with multiple joins, computes aggregations, and writes to a data lake. The pipeline takes 4 hours and costs $800/day in cloud compute. Your job as Principal Spark Architect: find every optimization opportunity, apply them in order, and measure the improvement at each step.

### 10-Step ETL Audit Framework
```
LAYER 1: READ
  [ ] 1. Read format (Parquet/ORC vs CSV/JSON)
  [ ] 2. Predicate pushdown to source (partition pruning, row group skipping)
  [ ] 3. Column pruning at scan (only read needed columns)

LAYER 2: TRANSFORM
  [ ] 4. Filter early (before joins, before aggregations)
  [ ] 5. Native functions only (no Python UDFs in hot paths)
  [ ] 6. Avoid unnecessary shuffles (sort, distinct, repartition)

LAYER 3: SHUFFLE
  [ ] 7. Shuffle partitions tuned to data size
  [ ] 8. Broadcast small tables (autoBroadcastJoinThreshold or hints)
  [ ] 9. Cache reused DataFrames (persist before fork points)

LAYER 4: WRITE
  [ ] 10. Write format (Parquet with compression), partition by, bucket by
```

In [ ]:
# ── Topic 7: Setup — write CSV and Parquet versions to compare ───────────────
import os, time

csv_path     = os.path.join(TEMP_DIR, "cabs_csv")
parquet_path = os.path.join(TEMP_DIR, "cabs_parquet")
output_path  = os.path.join(TEMP_DIR, "output")

# Build a larger dataset for this topic
cabs_t7 = cabs_raw
for _ in range(4):  # ~44,850 rows
    cabs_t7 = cabs_t7.union(cabs_raw)

# Write CSV (unoptimized source format)
cabs_t7.write.mode("overwrite").option("header","true").csv(csv_path)
# Write Parquet (optimized source format)
cabs_t7.write.mode("overwrite").parquet(parquet_path)

import glob
csv_size     = sum(os.path.getsize(f) for f in glob.glob(csv_path + "/*.csv"))
parquet_size = sum(os.path.getsize(f) for f in glob.glob(parquet_path + "/*.parquet"))
print(f"CSV size     : {csv_size/1024:.1f} KB")
print(f"Parquet size : {parquet_size/1024:.1f} KB")
print(f"Compression  : {csv_size/max(parquet_size,1):.1f}x smaller as Parquet")

### Problem Statement
The pipeline below contains 10 deliberate anti-patterns. Identify each one, apply the fix, and measure the improvement.

In [ ]:
# ── Topic 7: BAD ETL Pipeline — 10 anti-patterns combined ────────────────────

@udf(returnType=StringType())
def bad_normalize(name):
    # ANTI-PATTERN 5: Python UDF in hot path
    return name.strip().upper() if name else None

def run_bad_etl():
    # ANTI-PATTERN 1: Reading CSV instead of Parquet
    df = spark.read.option("header","true").csv(csv_path)

    # ANTI-PATTERN 2: select("*") — reads all 14 columns even when 4 are needed
    df = df.select("*")

    # ANTI-PATTERN 3: Filter AFTER join (not pushed down)
    joined = df.join(license_lookup, "LicenseType", "left")
    filtered = joined.filter(F.col("Active") == "YES")  # should be before join

    # ANTI-PATTERN 4: Using Python UDF
    enriched = filtered.withColumn("NormalizedName", bad_normalize(F.col("Name")))

    # ANTI-PATTERN 6: Redundant repartition(200) — forces unnecessary shuffle
    enriched = enriched.repartition(200)

    # ANTI-PATTERN 7: Recomputing the same DF twice (no cache)
    count_by_license = enriched.groupBy("LicenseType").count()
    avg_year_by_license = enriched.groupBy("LicenseType").avg("VehicleYear")
    # Both trigger full recomputation of 'enriched' — including CSV read + UDF + shuffle

    # ANTI-PATTERN 8: Joining large result to medium table without broadcast hint
    dim_table = spark.createDataFrame([
        ("OWNER MUST DRIVE","Active"),("NAMED DRIVER","Active"),("OWNER NAMED DRIVER","Legacy")
    ], ["LicenseType","Status"])
    final = enriched.join(dim_table, "LicenseType")  # dim_table not broadcast

    # ANTI-PATTERN 9: collect() on a potentially large result
    sample = final.collect()[:100]  # dangerous — collects all rows to driver first

    # ANTI-PATTERN 10: Writing as CSV instead of Parquet, no partitioning
    final.write.mode("overwrite").csv(output_path + "_bad")

    return len(sample)

t0 = time.time()
n = run_bad_etl()
t_bad = time.time() - t0
print(f"BAD pipeline: {t_bad:.2f}s (sampled {n} rows)")

### Interview Questions — Topic 7: End-to-End Pipeline Optimization

1. **Profiling from scratch:** Walk me through your process for profiling a slow Spark job you've never seen before. What do you look at first in the Spark UI? What do you look at second? What is your decision tree for identifying the root cause bottleneck?

2. **Format selection:** A team is migrating from CSV to Parquet. Describe ALL the performance advantages of Parquet over CSV for a Spark read workload. Include: column pruning, predicate pushdown, row group statistics, dictionary encoding, and compression codec selection.

3. **The `collect()` anti-pattern at scale:** A pipeline has `df.collect()` followed by Python processing. The DataFrame has 100K rows. Should you use `collect()`? At what row count does `collect()` become dangerous? What are the alternatives for common use cases?

4. **Reused DataFrame caching:** A DataFrame is used in 5 downstream operations. Should you always cache it? What are the costs of over-caching? Name a case where caching a reused DataFrame is SLOWER than re-computing it.

5. **Partition output optimization:** `df.write.partitionBy("date")` creates a directory per date. If you partition by `date` and `hour`, how many output directories are created for 30 days of data? What is the "small file problem" and how does it affect downstream readers?

6. **Bucketing vs partitioning:** Explain the difference between `partitionBy` (write-time) and `bucketBy` (table-level). For a table that is joined on `user_id` every day, which technique eliminates the join shuffle? What are the constraints for bucketing to work (matching bucket count, etc.)?

7. **AQE in end-to-end pipelines:** An ETL pipeline has 10 stages with shuffles between each. With AQE disabled, one misconfigured shuffle creates a bottleneck at stage 7. With AQE enabled, how does `coalesceShufflePartitions` handle this? Can AQE cause stage 7's output to feed stage 8 with wrong partition count?

8. **The `DISTINCT` trap:** `df.distinct()` triggers a full shuffle. A pipeline uses `distinct()` on a 10TB DataFrame to deduplicate. Describe three alternative approaches that are faster than `df.distinct()` for common deduplication patterns. When is `distinct()` unavoidable?

### Expected Plan Discussion — Topic 7

**BAD pipeline plan highlights:**
```
WriteCsv                               ← AP10: CSV output
+- SortMergeJoin [LicenseType]         ← AP8: no broadcast hint
   :- Exchange hashpartitioning(200)   ← AP6: repartition caused this exchange
   :  +- ArrowEvalPython [bad_normalize]  ← AP5: Python UDF
   :     +- Filter Active=YES          ← AP3: filter after join output
   :        +- SortMergeJoin           ← join with un-broadcast dim
   :           :- FileScan csv [*]     ← AP1: CSV, AP2: all columns
   :           +- LocalTableScan
   +- LocalTableScan dim
```
Count: 5+ Exchange nodes, 2 ArrowEvalPython nodes, CSV scan on all 14 columns.

**GOOD pipeline plan highlights:**
```
WriteParquet                           ← FIX10: Parquet output
+- *(2) BroadcastHashJoin              ← FIX8: broadcast dim
   :- *(2) Project [CabNumber,Name,LicenseType,Active,VehicleYear]
   :  +- *(2) Filter Active=YES        ← FIX3: filter before join
   :     +- *(2) BroadcastHashJoin     ← FIX8: broadcast license_lookup
   :        :- *(2) FileScan parquet [CabNumber,Name,LicenseType,Active,VehicleYear]
   :        :  (FIX1: Parquet, FIX2: only 5 cols, FIX4: early filter)
   :        +- BroadcastExchange license_lookup
   +- BroadcastExchange dim
```
No ArrowEvalPython, no unnecessary Exchange, Parquet scan with column pruning.
Cache point is between the join and the two aggregations (avoiding recomputation).

### Spark UI Investigation Guide — Pipeline Profiling Process

**Step 1: Jobs Tab — find the bottleneck job**
- Sort jobs by duration descending. The longest job is your first target.
- Check for failed jobs (red) — any failure wastes all previous computation.
- Look for jobs with many stages — each stage boundary = shuffle = potential bottleneck.

**Step 2: Stages Tab — find the bottleneck stage**
- Sort stages by duration descending.
- Click the longest stage → look at:
  - `Input Size / Records` — how much data was read?
  - `Shuffle Read Size / Records` — how much shuffle data?
  - `GC Time` — what fraction of task time is GC?
  - `Task Duration` distribution — is there a straggler (long tail)?

**Step 3: Task Detail — identify the root cause**
For a straggler (one slow task):
  - `Input Size` of the slow task >> median → data skew (one large partition)
  - `GC Time` of the slow task >> median → memory pressure in that partition
  - `Shuffle Read Size` >> median → one shuffle partition received too many keys (skew)

For all-tasks-slow:
  - High GC across all tasks → executor memory undersized
  - High Shuffle Read everywhere → wrong shuffle partition count
  - Long task deserialization → UDF closures too large

**Step 4: SQL Tab — validate the plan**
- Look for unexpected Exchange nodes
- Look for ArrowEvalPython nodes (Python UDFs)
- Look for missing `*(N)` WSCG markers
- Check statistics annotations for `8.0 EiB` (missing CBO stats)

**Step 5: Storage Tab — validate caching**
- Are cached DataFrames actually cached? (shows fraction cached)
- Is storage memory nearly full? (eviction happening)
- Is `Memory Deserialized` or `Memory Serialized`? (serialized = more GC-friendly)

In [ ]:
# ── Topic 7: OPTIMIZED ETL Pipeline — all 10 fixes applied ──────────────────
from pyspark import StorageLevel

def run_good_etl():
    # FIX 1: Read Parquet (columnar, compressed, predicate-pushdown capable)
    df = spark.read.parquet(parquet_path)

    # FIX 2 + FIX 3 + FIX 4: Select only needed columns AND filter early
    df = (df
        .select("CabNumber", "Name", "LicenseType", "Active", "VehicleYear")
        .filter(F.col("Active") == "YES")
    )

    # FIX 5: Native Spark function instead of UDF
    dim_table = spark.createDataFrame([
        ("OWNER MUST DRIVE","Active"),
        ("NAMED DRIVER","Active"),
        ("OWNER NAMED DRIVER","Legacy")
    ], ["LicenseType","Status"])

    # FIX 8: Broadcast small dimension tables
    enriched = (df
        .join(F.broadcast(license_lookup.select("LicenseType","LicenseCode")),
              "LicenseType", "left")
        .join(F.broadcast(dim_table), "LicenseType", "left")
        .withColumn("NormalizedName",             # FIX 5: no UDF
            F.upper(F.trim(F.col("Name"))))
    )

    # FIX 9: Cache before fork point (two downstream aggregations reuse enriched)
    enriched.persist(StorageLevel.MEMORY_AND_DISK)
    enriched.count()  # materialize the cache

    # FIX 6: No repartition — partitions are already appropriate
    # Both aggregations now read from cache (no recomputation)
    count_by_license   = enriched.groupBy("LicenseType").count()
    avg_year_by_license = enriched.groupBy("LicenseType").avg("VehicleYear")

    # FIX 9b: Use limit() + collect() instead of collect() on full result
    sample = enriched.limit(100).collect()  # only 100 rows go to driver

    # FIX 10: Write as Parquet, partitioned by LicenseType
    enriched.write.mode("overwrite") \
        .partitionBy("LicenseType") \
        .parquet(output_path + "_good")

    enriched.unpersist()  # release cache when no longer needed
    return len(sample)

t0 = time.time()
n = run_good_etl()
t_good = time.time() - t0
print(f"GOOD pipeline: {t_good:.2f}s (sampled {n} rows)")
print(f"BAD pipeline : {t_bad:.2f}s")
print(f"Speedup      : {t_bad/max(t_good,0.001):.2f}x")

### Production Best Practices — End-to-End Pipeline Optimization

1. **Always read Parquet/ORC, never CSV/JSON in production pipelines.** CSV requires full scan of every row to parse; Parquet enables column skipping and row-group statistics. A 5TB CSV file read becomes a 500GB Parquet read after column pruning and predicate pushdown — 10× less I/O.

2. **Filter before join, always.** Even though Catalyst will push predicates down, write it explicitly. Not all data sources support Catalyst pushdown (JDBC, custom connectors). Explicit early filtering is always safe and never wrong.

3. **Identify fork points and cache them.** Any DataFrame that feeds 2+ downstream operations should be cached with `persist(StorageLevel.MEMORY_AND_DISK)`. Call `.count()` to materialize the cache immediately. Call `.unpersist()` when done. Uncached fork points double (or triple) your computation cost.

4. **`limit(N).collect()` instead of `collect()`.** If you need a sample, use `limit()` first. `collect()` on an unbounded DataFrame sends all rows to the driver — one day this will be a 500GB DataFrame and your driver will OOM. Make it a team coding standard: `collect()` is always preceded by `limit()`.

5. **`partitionBy` output with 2-4 high-cardinality columns maximum.** More partition columns = exponentially more output directories = small file problem. Target: each output file should be 128MB-1GB. If `partitionBy("date","hour","region")` creates 10,000 directories with 1KB files each, downstream readers suffer enormously.

6. **`coalesce(N)` before writing to control output file count.** After a shuffle-heavy pipeline, you may have 2,000 small partitions. `df.coalesce(50).write.parquet(...)` merges them without a shuffle. Use `coalesce` (not `repartition`) when reducing partition count — it avoids the extra shuffle.

7. **Avoid `orderBy` / `sort` unless you explicitly need globally sorted output.** `orderBy` triggers a full shuffle to collect all rows into a single sort order. In most ETL pipelines, you don't need global order — use `sortWithinPartitions` if you need local order for downstream processing.

8. **`dropDuplicates(subset=[...])` is faster than `distinct()`.** `distinct()` compares all columns. `dropDuplicates(["id","date"])` compares only the key columns. For a 200-column DataFrame, this reduces shuffle data by 99%. Also consider `Window.row_number()` dedup for row-retention control.

9. **Write with `snappy` compression for Parquet.** Snappy is the sweet spot: fast decompression (important for Spark reads), decent compression ratio (3-5×). Use `zstd` when storage cost dominates and you can afford slightly slower decompression. Never use `gzip` for frequently-read Spark data.

10. **Instrument every pipeline with timing checkpoints.** Wrap each major step in `time.time()` and log the duration. When a pipeline regresses, you instantly know which step slowed down. Without instrumentation, you are flying blind — every optimization is a guess.

### Follow-up Questions — Topic 7

- A Delta Lake `MERGE INTO` (upsert) operation is slow on a 100GB table. Walk through the specific performance challenges of MERGE and the tuning strategies (file compaction, Z-ORDER, bloom filters) that address them.
- Your pipeline uses `df.checkpoint()` instead of `df.cache()`. What is the difference? When is checkpointing better than caching? What are the two types of checkpointing in Spark and when do you use each?
- How does Spark handle writing to S3 vs HDFS differently? What is the S3A committer and why does it exist? What is the `_SUCCESS` file and what problems does S3's eventual consistency create for Spark writes?

In [ ]:
# ── Topic 7: Expert Solution — Pipeline Audit Scorecard ─────────────────────

def audit_etl_pipeline(df_or_plan_str, label="Pipeline"):
    """
    Scores an ETL pipeline against the 10-step optimization checklist.
    Returns a scorecard that can be used in code review.
    """
    if hasattr(df_or_plan_str, "queryExecution"):
        plan = str(df_or_plan_str.queryExecution.executedPlan)
    else:
        plan = str(df_or_plan_str)

    checks = []

    # Check 1: Source format
    if "FileScan csv" in plan:
        checks.append(("FAIL", "Read Layer",    "Reading CSV — migrate to Parquet/ORC"))
    elif "FileScan parquet" in plan or "FileScan orc" in plan:
        checks.append(("PASS", "Read Layer",    "Reading Parquet/ORC"))
    else:
        checks.append(("INFO", "Read Layer",    "Source format unknown"))

    # Check 5: Python UDFs
    udf_count = plan.count("ArrowEvalPython") + plan.count("BatchEvalPython")
    if udf_count > 0:
        checks.append(("FAIL", "Transform Layer",
                        f"{udf_count} Python UDF boundary(ies) — rewrite with native functions"))
    else:
        checks.append(("PASS", "Transform Layer", "No Python UDFs detected"))

    # Check 8: Broadcast joins
    bhj_count = plan.count("BroadcastHashJoin")
    smj_count = plan.count("SortMergeJoin")
    if smj_count > 0 and bhj_count == 0:
        checks.append(("WARN", "Shuffle Layer",
                        f"{smj_count} SortMergeJoin(s) — check if any side can be broadcast"))
    elif bhj_count > 0:
        checks.append(("PASS", "Shuffle Layer",
                        f"{bhj_count} BroadcastHashJoin(s) — good join strategy"))
    else:
        checks.append(("INFO", "Shuffle Layer", "No joins detected"))

    # Check 6: Exchange count
    exchange_count = plan.count("Exchange")
    broadcast_count = plan.count("BroadcastExchange")
    shuffle_count = exchange_count - broadcast_count
    if shuffle_count > 3:
        checks.append(("WARN", "Shuffle Layer",
                        f"{shuffle_count} shuffle Exchange(s) — high shuffle count"))
    elif shuffle_count > 0:
        checks.append(("PASS", "Shuffle Layer",
                        f"{shuffle_count} shuffle Exchange(s) — acceptable"))

    # Check: WSCG coverage
    if plan.count("WholeStageCodegen") == 0 and udf_count == 0:
        checks.append(("WARN", "Tungsten",  "No WholeStageCodegen — check codegen.maxFields"))
    elif udf_count == 0:
        checks.append(("PASS", "Tungsten",  "WholeStageCodegen active"))

    # Score
    pass_count = sum(1 for s, _, _ in checks if s == "PASS")
    fail_count = sum(1 for s, _, _ in checks if s == "FAIL")
    warn_count = sum(1 for s, _, _ in checks if s == "WARN")
    score = int(100 * pass_count / max(1, pass_count + fail_count + warn_count * 0.5))

    print(f"\n{'='*60}")
    print(f"ETL Pipeline Audit: {label}   Score: {score}/100")
    print(f"{'='*60}")
    for status, layer, message in checks:
        icon = "OK  " if status == "PASS" else ("FAIL" if status == "FAIL" else "WARN")
        print(f"  [{icon}] {layer:<18} {message}")

# Audit both pipelines
print("Auditing bad ETL pipeline plan:")
bad_df = (spark.read.option("header","true").csv(csv_path)
          .join(license_lookup, "LicenseType", "left")
          .filter(F.col("Active") == "YES"))
audit_etl_pipeline(bad_df, "BAD Pipeline")

print("\nAuditing good ETL pipeline plan:")
good_df = (spark.read.parquet(parquet_path)
           .select("CabNumber","Name","LicenseType","Active","VehicleYear")
           .filter(F.col("Active") == "YES")
           .join(F.broadcast(license_lookup.select("LicenseType","LicenseCode")), "LicenseType","left"))
audit_etl_pipeline(good_df, "GOOD Pipeline")

## Topic 8 — Spark UI Deep Dive

### Scenario
A junior engineer pages you at 2am: "The daily pipeline is stuck. It's been running for 5 hours, usually takes 45 minutes. I can see it's still running in the Spark UI but I don't know what to look at." You walk them through every tab of the Spark UI while they screen-share.

### Spark UI Architecture
```
Spark UI runs on the Driver node: http://<driver-host>:4040
After job completes: available via History Server at :18080

TABS:
  /jobs        — Job-level overview, duration, status
  /stages      — Stage-level breakdown, shuffle metrics, task counts
  /storage     — Cached RDDs/DataFrames, memory usage per partition
  /environment — Spark config, JVM flags, classpath
  /executors   — Per-executor metrics: CPU, GC, shuffle I/O
  /SQL/        — DataFrame query plans, DAG visualization
  /streaming   — (Structured Streaming only) micro-batch metrics
```

### Complete Metric Reference Table

| Metric | Location | Healthy | Problem Indicator | Root Cause |
|--------|----------|---------|-------------------|------------|
| GC Time % | Stages tab | <10% | >20% | Executor undersized, UDFs, large objects |
| Shuffle Read/Write | Stages tab | Proportional to data | Too large vs input | Wrong shuffle.partitions |
| Shuffle Spill (Disk) | Stages tab | 0 | >0 | Execution memory too small |
| Task Duration P99/P50 | Stage detail | <2× | >5× | Data skew |
| Peak Execution Memory | Stage detail | <80% executor | Near executor limit | OOM risk |
| Task Deserialization | Task detail | <100ms | >1s | Large broadcast or UDF closure |
| Result Serialization | Task detail | <100ms | >1s | collect() returning too much data |
| Input vs Output rows | Stage detail | Varies | Output >> Input | Explode or cartesian |
| Executor Lost | Jobs tab | 0 | Any | OOM, network, container kill |
| Failed Stages | Stages tab | 0 | Any | Check error message |

In [ ]:
# ── Topic 8: Setup — create a scenario with a skewed partition ───────────────
# Create the 199 fast + 1 slow task scenario from the interview question

# Normal data: 199 rows with even distribution
normal_data = [(str(i), "LicenseType_" + str(i % 3), 2015 + i % 10)
               for i in range(1990)]

# Skewed data: 10x more rows in one partition
skewed_data = [("SKEWED_" + str(i), "OWNER MUST DRIVE", 2018)
               for i in range(9000)]

all_data = normal_data + skewed_data
df_skewed = spark.createDataFrame(all_data, ["CabNumber", "LicenseType", "VehicleYear"])
df_skewed.createOrReplaceTempView("skewed_cabs")

print(f"Total rows: {df_skewed.count():,}")
print(f"Distribution by LicenseType:")
df_skewed.groupBy("LicenseType").count().orderBy(F.desc("count")).show(5)
print()
print("With spark.sql.shuffle.partitions=10 and skew on 'OWNER MUST DRIVE',")
print("one partition will have ~9,000 rows while others have ~660 rows each.")
print("This creates the 199 fast + 1 slow task scenario.")

### Problem Statement
In the Spark UI, you see 199 tasks finishing in 2 seconds and 1 task taking 4 minutes. What is the root cause? Walk through your complete investigation and fix.

In [ ]:
# ── Topic 8: Demonstrate data skew — observe task duration distribution ───────
spark.conf.set("spark.sql.shuffle.partitions", "10")
spark.conf.set("spark.sql.adaptive.enabled", "false")  # see skew without AQE fix

import time

# This aggregation will create a skewed partition
t0 = time.time()
result = df_skewed.groupBy("LicenseType").agg(
    F.count("*").alias("cnt"),
    F.avg("VehicleYear").alias("avg_year"),
    F.max("CabNumber").alias("max_cab")
)
result.collect()
t_skewed = time.time() - t0
print(f"Skewed aggregation: {t_skewed:.2f}s")
print()
print("In the Spark UI Stages tab for this job, look for:")
print("  - Task duration: some tasks 2s, one task MUCH longer")
print("  - Input records: one task has ~9,000, others have ~660")
print("  - The slow task's partition contains all 'OWNER MUST DRIVE' rows")
print()

# Now fix with AQE skew handling
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "2")

t0 = time.time()
result_aqe = df_skewed.groupBy("LicenseType").agg(
    F.count("*").alias("cnt"),
    F.avg("VehicleYear").alias("avg_year"),
    F.max("CabNumber").alias("max_cab")
)
result_aqe.collect()
t_aqe = time.time() - t0
print(f"AQE-fixed aggregation: {t_aqe:.2f}s")
print(f"Speedup: {t_skewed/max(t_aqe,0.001):.2f}x")

# Reset
spark.conf.set("spark.sql.shuffle.partitions", "200")

### Interview Questions — Topic 8: Spark UI Deep Dive

1. **The straggler investigation (the scenario above):** In the Stages tab, you see 199 tasks finishing in 2 seconds and 1 task taking 4 minutes. Walk me through your complete investigation: which columns do you look at in the stage detail? How do you confirm it is data skew vs GC vs network vs CPU? What are the three ways to fix data skew depending on where it occurs?

2. **SQL DAG reading:** In the SQL tab's DAG visualization, you see: `Scan Parquet → Filter → BroadcastExchange → BroadcastHashJoin → HashAggregate → Exchange → HashAggregate`. Explain each node: what does it do, what stage boundary does it create, and what metrics would you look at for each one?

3. **Timeline view:** The Spark UI's Jobs tab shows a timeline. You see a 10-minute gap between Stage 4 completion and Stage 5 start. What could cause this gap? Name five possible causes and how you distinguish between them.

4. **Executor tab investigation:** You see one executor with 3× the GC time of all others. What are the possible root causes? How do you confirm which executor is holding a cached partition that is causing the GC pressure? How do you force Spark to evict it?

5. **Storage tab:** A DataFrame is cached but the Storage tab shows "Fraction Cached: 45%". What does this mean? Is the pipeline correct? What happens when Spark tries to read a partition that was evicted from the cache? How does this affect job duration and predictability?

6. **Environment tab forensics:** A job that worked last week now runs 3× slower. The code is unchanged. How do you use the Environment tab to detect if: (a) a Spark config changed, (b) a JVM flag was removed, (c) a library version changed, (d) the cluster was resized?

7. **History Server and Event Logs:** The Spark UI is unavailable (driver already exited). How do you access historical UI data? What is the Spark event log? Where is it stored? What information is preserved vs lost after the driver exits?

8. **Metrics deep dive — the 4-minute task:** In a stage with 200 tasks, the slow task has: `Duration=4min, GC Time=10s, Shuffle Read=500MB, Input=2GB, Deserialize=5ms`. Based only on these numbers, what is the most likely root cause? Explain your reasoning using each metric value.

### Expected Logical Plan Discussion — Topic 8

The Spark UI SQL tab shows four plan representations (from the "Details" link):

**1. Parsed Logical Plan** — raw AST, no resolution
```
'Aggregate ['LicenseType], ['LicenseType, count(1) AS cnt, avg('VehicleYear) AS avg_year]
+- 'UnresolvedRelation [skewed_cabs]
```

**2. Analyzed Logical Plan** — column references resolved
```
Aggregate [LicenseType#12], [LicenseType#12, count(1) AS cnt#45L, avg(VehicleYear#14) AS avg_year#46]
+- SubqueryAlias skewed_cabs
   +- LocalRelation [CabNumber#10, LicenseType#12, VehicleYear#14]
```

**3. Optimized Logical Plan** — Catalyst rules applied
```
Aggregate [LicenseType], [LicenseType, count(1) AS cnt, avg(VehicleYear) AS avg_year, max(CabNumber) AS max_cab]
+- LocalRelation [CabNumber, LicenseType, VehicleYear]
```
(ColumnPruning removed any unused columns; ConstantFolding simplified expressions)

**4. Physical Plan** — SparkPlanner output
```
AdaptiveSparkPlan isFinalPlan=false            ← AQE wrapper
+- HashAggregate (partial=false)               ← reduce side aggregate
   +- Exchange hashpartitioning(LicenseType, 10) ← shuffle
      +- HashAggregate (partial=true)           ← map side partial aggregate
         +- LocalTableScan [CabNumber, LicenseType, VehicleYear]
```
**Two-phase aggregation:** partial aggregate on map side (before shuffle) reduces data volume sent to shuffle. Full aggregate on reduce side. This is why `HashAggregate` appears twice in the physical plan.

### Spark UI Investigation Guide — Complete Tab-by-Tab Reference

---
**JOBS TAB** (`/jobs`)
Purpose: Overview of all Spark actions triggered since application start.
What to look for:
- Failed jobs (red) — click to see the failed stage and error
- Job duration outliers — one job taking 10× longer than others
- Active jobs that never complete — stuck task or deadlock
- Total jobs count — very high number may indicate a `count()` in a loop (anti-pattern)

---
**STAGES TAB** (`/stages`)
Purpose: Breakdown of each stage within each job.
Critical columns:
- `Duration` — wall time for the stage (dominated by the slowest task)
- `Input Size` — data read from disk/cache
- `Output Size` — data written to shuffle/output
- `Shuffle Read` — data received from previous stage's shuffle write
- `Shuffle Write` — data written for next stage to read
- `GC Time` — JVM GC time summed across all tasks in the stage

**Stage Detail** (click any stage):
- **Task Duration histogram:** bimodal = skew, long tail = straggler
- **Shuffle Spill (Memory/Disk):** any disk spill = execution memory too small
- **Task timeline:** visualizes which executor ran which task, when

---
**STORAGE TAB** (`/storage`)
Purpose: Shows all currently cached RDDs/DataFrames.
- Fraction Cached < 100%: some partitions evicted → re-read from source on access
- Memory Used: if near total cluster storage memory → eviction likely
- `Memory Deserialized` vs `Memory Serialized`: deserialized is faster to access (no deserialize cost) but uses more memory

---
**ENVIRONMENT TAB** (`/environment`)
Purpose: Complete configuration snapshot at job start time.
Most important sections:
- **Spark Properties:** all `spark.*` configs — check for wrong shuffle.partitions, disabled AQE, incorrect memory settings
- **JVM Information:** GC flags, heap size (-Xmx), extra Java options
- **Classpath Entries:** dependency JARs — check for version conflicts

---
**EXECUTORS TAB** (`/executors`)
Purpose: Per-executor resource utilization.
Key columns:
- `Task Time` / `GC Time` — compute GC% per executor
- `Storage Memory` — cached data per executor
- `Shuffle Read/Write` — I/O per executor (should be roughly equal across executors; imbalance = skew)
- `Blacklisted` — executors that Spark stopped using due to repeated failures
Sort by `GC Time` descending to find struggling executors immediately.

---
**SQL/DATAFRAME TAB** (`/SQL`)
Purpose: Query plan visualization for DataFrame/SQL operations.
- Click any query → see DAG boxes
- Each box = one physical operator
- Arrows between boxes = data flow
- DAG **node color**: dark = actively running, light = complete
- `Exchange` boxes = shuffle points = potential bottlenecks
- `WholeStageCodegen` labels = WSCG active = good
- Click "Details" → see all 4 plan phases (Parsed/Analyzed/Optimized/Physical)
- Statistics shown: `numOutputRows`, `dataSize`, `peakMemory` per operator

In [ ]:
# ── Topic 8: Optimization Exercise — demonstrate skew fix approaches ─────────
spark.conf.set("spark.sql.shuffle.partitions", "10")

# APPROACH 1: AQE skew join (automatic, requires AQE)
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

# APPROACH 2: Salt key to distribute skewed values manually
# Add a random salt (0-4) to break the skewed partition into 5 smaller ones
df_salted = df_skewed.withColumn(
    "salt",
    (F.hash(F.col("CabNumber")) % 5).cast("string")
)
df_salted_agg = (df_salted
    .groupBy("LicenseType", "salt")
    .agg(F.count("*").alias("cnt_partial"),
         F.sum("VehicleYear").alias("year_sum"),
         F.max("CabNumber").alias("max_cab"))
    .groupBy("LicenseType")
    .agg(F.sum("cnt_partial").alias("cnt"),
         (F.sum("year_sum") / F.sum("cnt_partial")).alias("avg_year"),
         F.max("max_cab").alias("max_cab"))
)

t0 = time.time()
salted_result = df_salted_agg.collect()
t_salted = time.time() - t0

# APPROACH 3: Filter-and-union for highly skewed key
df_normal_keys = df_skewed.filter(F.col("LicenseType") != "OWNER MUST DRIVE")
df_skewed_key  = df_skewed.filter(F.col("LicenseType") == "OWNER MUST DRIVE")

agg_normal = df_normal_keys.groupBy("LicenseType").agg(
    F.count("*").alias("cnt"), F.avg("VehicleYear").alias("avg_year"))
agg_skewed = df_skewed_key.groupBy("LicenseType").agg(
    F.count("*").alias("cnt"), F.avg("VehicleYear").alias("avg_year"))

t0 = time.time()
filter_union_result = agg_normal.union(agg_skewed).collect()
t_filter_union = time.time() - t0

print("Skew Fix Approaches:")
print(f"  AQE (automatic)     : Already benchmarked above ({t_aqe:.2f}s)")
print(f"  Salt key (manual)   : {t_salted:.2f}s  ({len(salted_result)} groups)")
print(f"  Filter+union        : {t_filter_union:.2f}s  ({len(filter_union_result)} groups)")
print()
print("Choose based on:")
print("  AQE: use for join skew — handles runtime, needs no code change")
print("  Salt: use for GROUP BY on known-skewed key, especially without AQE")
print("  Filter+union: use when ONE key is disproportionately large (>50% of data)")

spark.conf.set("spark.sql.shuffle.partitions", "200")

### Production Best Practices — Spark UI Usage

1. **Make the Spark UI your first stop, not the logs.** Logs show you what failed. The UI shows you WHY it's slow. The Stages tab with its shuffle metrics tells you more in 30 seconds than 30 minutes of log parsing.

2. **Set up Spark History Server in all environments.** The UI disappears when the driver exits. History Server (`spark.history.fs.logDirectory`) preserves all UI data permanently. Run it as a persistent service. Without it, you cannot post-mortem a production incident.

3. **Enable event logging: `spark.eventLog.enabled=true`.** Event logs are the raw data for the History Server. Set `spark.eventLog.dir` to a durable location (HDFS, S3, GCS). Retention: keep 30 days minimum for incident investigation.

4. **Use the SQL tab's DAG to count Exchange nodes before every deploy.** A regression that adds an unexpected shuffle = one more Exchange node. This is the fastest single indicator of a performance regression. Add plan Exchange-count to your CI pipeline as a regression test.

5. **The 10% rule for GC:** Any stage where `GC Time > 10% × Task Time` needs investigation. Any stage where `GC Time > 20%` needs immediate remediation. These are thresholds you should memorize — they come up in every Spark performance interview.

6. **Shuffle Spill (Disk) > 0 is always a bug.** When Spark spills shuffle data to disk, it means execution memory was insufficient. The disk I/O is 10-100× slower than in-memory operations. Fix: increase `spark.sql.shuffle.partitions` to reduce per-partition size, or increase executor memory.

7. **The straggler task is the job's bottleneck.** A stage with 1 task at 10% of the stage's total time can be ignored. A stage with 1 task at 90% of the stage's time is the only bottleneck — fixing it will reduce the stage time by 9×. Always look at the task duration distribution, not just the mean.

8. **`spark.speculation=true` can mask skew.** Task speculation launches a duplicate of slow tasks on a different executor. This hides skew in the UI (the speculative copy completes, the original is killed). Skew is hidden but not fixed — you are wasting 2× resources on the skewed partition. Fix skew; don't speculate around it.

9. **Monitor the Storage tab for cache eviction patterns.** "Fraction Cached: 45%" means Spark is continuously evicting and re-caching partitions — a sign of insufficient storage memory. The job runs but slowly, with unpredictable re-reads. Either add memory, use `MEMORY_AND_DISK`, or reduce the cached DataFrame's scope.

10. **In the Executors tab, equal shuffle read/write across executors = no skew.** If one executor has 10× the shuffle read of others, that executor is processing the skewed partition. You can now identify the exact executor, its host machine, and the GC logs to analyze — all from the Executors tab alone.

### Follow-up Questions — Topic 8

- Spark 3.4+ introduced the `spark.sql.ui.retainedExecutions` config. What happens when the SQL tab fills up with thousands of queries (e.g., in a loop)? How does this affect driver memory? What config prevents driver OOM from query history accumulation?
- What is the `spark.metrics.conf` file and how do you export Spark metrics to Prometheus/Grafana? What additional metrics does a metrics sink expose that the Spark UI does not show (hint: JVM buffer pool metrics, network I/O bytes)?
- In a Structured Streaming application, the Spark UI shows a `Streaming` tab with micro-batch durations. What does `Trigger Processing Time > Trigger Interval` indicate? How do you use the streaming UI to diagnose source backpressure vs processing bottleneck?

In [ ]:
# ── Topic 8: Expert Solution — Spark UI Programmatic Health Check ─────────────
# Access live Spark UI metrics via the REST API (port 4040)
# In production: integrate this into your monitoring pipeline

import json
try:
    import urllib.request
    UI_BASE = f"http://localhost:{spark.sparkContext.uiWebUrl.split(':')[-1]}"

    def spark_ui_get(endpoint):
        url = f"{UI_BASE}/api/v1/applications/{spark.sparkContext.applicationId}{endpoint}"
        try:
            with urllib.request.urlopen(url, timeout=2) as r:
                return json.loads(r.read())
        except Exception:
            return None

    # Get executor metrics
    executors = spark_ui_get("/executors")
    if executors:
        print(f"Active executors: {len([e for e in executors if e.get('isActive')])}")
        for e in executors[:3]:
            if e.get("isActive"):
                task_time  = e.get("totalTasksDuration", 0)
                gc_time    = e.get("totalGCTime", 0)
                gc_pct     = (gc_time / max(task_time, 1)) * 100
                print(f"  Executor {e['id']}: "
                      f"tasks={e.get('totalTasks',0)}, "
                      f"GC={gc_pct:.1f}%, "
                      f"shuffle_read={e.get('totalShuffleRead',0)//1024}KB, "
                      f"storage_mem={e.get('memoryUsed',0)//1024//1024}MB used")
    else:
        print("Spark UI REST API: could not connect (normal in local mode without active jobs)")
        print("In cluster mode, use: http://<driver>:4040/api/v1/applications/<appId>/executors")

except Exception as e:
    print(f"UI API not available: {e}")

print()
print("=== Manual Spark UI Checklist (for interviews) ===")
checklist = [
    ("Jobs tab",      "Any failed jobs? Any active job stuck for >5× expected duration?"),
    ("Stages tab",    "Sort by Duration. Is top stage > 50% of job time? Click it."),
    ("Stage detail",  "Check: GC%, Shuffle Spill, Task Duration histogram (bimodal=skew)"),
    ("SQL tab",       "Count Exchange nodes. Any ArrowEvalPython? Missing *(N) WSCG?"),
    ("Executors tab", "Sort by GC Time. Any executor with >20% GC? Shuffle imbalance?"),
    ("Storage tab",   "Fraction Cached < 100%? Memory Used near limit?"),
    ("Environment",   "AQE enabled? CBO enabled? shuffle.partitions correct? GC flags set?"),
]
for tab, check in checklist:
    print(f"  [{tab:<15}] {check}")

---
## Notebook Summary — Expert Spark Interview Preparation

| Topic | Core Concept | Key Interview Hook |
|-------|-------------|-------------------|
| 1. Catalyst Optimizer | 4-phase pipeline, TreeNode, Rule batches | "Explain PredicatePushdown limits" |
| 2. Tungsten Engine | WSCG, UnsafeRow, sun.misc.Unsafe | "Why does a UDF break WholeStageCodegen?" |
| 3. Cost-Based Optimizer | ANALYZE TABLE, NDV, DP join reorder | "What does CBO estimate when stats are missing?" |
| 4. Join Strategy Selection | 5 strategies, JoinSelection, AQE conversion | "When would you choose ShuffleHashJoin over SMJ?" |
| 5. Cluster Sizing | 5-core rule, memory formula, parallelism | "Design a cluster for 10TB in 30 minutes" |
| 6. Executor Tuning | Fat vs thin, G1GC, off-heap | "What causes 55% GC overhead?" |
| 7. Pipeline Optimization | 10-step audit, format, cache, broadcast | "Walk me through profiling a slow job" |
| 8. Spark UI Deep Dive | Every tab, every metric, skew diagnosis | "199 tasks take 2s, 1 takes 4 minutes. Why?" |

### Key Production Configs (Definitive Reference)
```python
# Minimum viable production Spark config:
spark = SparkSession.builder \
    .config("spark.sql.adaptive.enabled",                "true") \
    .config("spark.sql.adaptive.skewJoin.enabled",       "true") \
    .config("spark.sql.cbo.enabled",                     "true") \
    .config("spark.sql.cbo.joinReorder.enabled",         "true") \
    .config("spark.sql.statistics.histogram.enabled",    "true") \
    .config("spark.sql.shuffle.partitions",              "400")   # tune to total_cores × 3
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.executor.extraJavaOptions",
            "-XX:+UseG1GC -XX:G1HeapRegionSize=32m -XX:MaxGCPauseMillis=200") \
    .config("spark.eventLog.enabled",                    "true") \
    .config("spark.memory.offHeap.enabled",              "true") \
    .config("spark.memory.offHeap.size",                 "5368709120")  # 5GB
    .getOrCreate()
```

### Spark Source Code References for Deep Discussions
- `org.apache.spark.sql.catalyst.optimizer.Optimizer` — rule batch application
- `org.apache.spark.sql.catalyst.trees.TreeNode` — AST foundation
- `org.apache.spark.sql.execution.SparkPlanner` — logical→physical
- `org.apache.spark.sql.execution.joins.JoinSelection` — join strategy selection
- `org.apache.spark.sql.execution.WholeStageCodegenExec` — WSCG entry point
- `org.apache.spark.unsafe.types.UTF8String` — Tungsten string representation
- `org.apache.spark.sql.execution.UnsafeExternalSorter` — off-heap sort
- `org.apache.spark.sql.adaptive.AdaptiveSparkPlanExec` — AQE runtime replanning